In [ ]:
!pip install optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 364.4/364.4 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 233.5/233.5 kB 12.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.6/78.6 kB 5.3 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error
from sklearn.decomposition import PCA
from sklearn.preprocessing import LabelEncoder
import optuna

In [ ]:
data = pd.read_csv('train_feature_v5.csv')
test_df = pd.read_csv('test_feature_v5.csv')

In [ ]:
pca_cols_train = data.iloc[:,5:27]

le = LabelEncoder()
pca_cols_train = pca_cols_train.apply(le.fit_transform)

data = data.drop(columns=pca_cols_train.columns)
data = pd.concat([data, pca_cols_train], axis=1)

pca_cols_test = test_df.iloc[:,5:27]

le = LabelEncoder()
pca_cols_test = pca_cols_test.apply(le.fit_transform)

test_df = test_df.drop(columns=pca_cols_train.columns)
test_df = pd.concat([test_df, pca_cols_test], axis=1)

In [ ]:
pca_cols_train = data.iloc[:,7:13]

le = LabelEncoder()
pca_cols_train = pca_cols_train.apply(le.fit_transform)

data = data.drop(columns=pca_cols_train.columns)
data = pd.concat([data, pca_cols_train], axis=1)

pca_cols_test = test_df.iloc[:,7:13]

le = LabelEncoder()
pca_cols_test = pca_cols_test.apply(le.fit_transform)

test_df = test_df.drop(columns=pca_cols_train.columns)
test_df = pd.concat([test_df, pca_cols_test], axis=1)

In [ ]:
X = data.drop(columns=["price"]).values  # Exclude target variable for input
y = data["price"].values  # Target variable

In [ ]:
# Normalize features and target variable
scaler_X = StandardScaler()
X_normalized = scaler_X.fit_transform(X)
scaler_y = StandardScaler()
y_normalized = scaler_y.fit_transform(y.reshape(-1, 1)).flatten()

In [ ]:
# Split into training and validation sets
X_train, X_val, y_train, y_val = train_test_split(X_normalized, y_normalized, test_size=0.2, random_state=42)

# Convert to PyTorch tensors
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.float32).view(-1, 1)
X_val_tensor = torch.tensor(X_val, dtype=torch.float32)
y_val_tensor = torch.tensor(y_val, dtype=torch.float32).view(-1, 1)

# Create DataLoaders for training and validation
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
val_dataset = TensorDataset(X_val_tensor, y_val_tensor)
# train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
# val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

In [ ]:
# Training function with early stopping
def train_model(model, train_loader, val_loader, criterion, optimizer, epochs=20):
    best_val_loss, epochs_no_improve = float('inf'), 0 # Initialize best_val_loss to infinity
    for epoch in range(epochs):
        model.train()
        train_loss = 0.0
        for inputs, targets in train_loader:
            inputs, targets = inputs.to(device), targets.to(device) # Move inputs and targets to device
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            loss.backward()
            optimizer.step()
            train_loss += loss.item() * inputs.size(0)

        # Validation phase
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for inputs, targets in val_loader:
                inputs, targets = inputs.to(device), targets.to(device) # Move inputs and targets to device
                outputs = model(inputs)
                loss = criterion(outputs, targets)
                val_loss += loss.item() * inputs.size(0)

        # Calculate average losses
        train_loss /= len(train_loader.dataset)
        val_loss /= len(val_loader.dataset)

        print(f"Epoch {epoch+1}/{epochs}, Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}")

        # Early stopping check
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            epochs_no_improve = 0
            best_model = model.state_dict()  # Save the best model's parameters
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= patience:
                print(f"Early stopping triggered after {epoch+1} epochs.")
                model.load_state_dict(best_model)  # Load the best model parameters
                break

    # Validation phase
    model.eval()
    val_targets, val_predictions = [], []
    with torch.no_grad():
        for inputs, targets in val_loader:
            inputs, targets = inputs.to(device), targets.to(device) # Move inputs and targets to device
            outputs = model(inputs)
            val_predictions.append(outputs.cpu().numpy())
            val_targets.append(targets.cpu().numpy())

    # Calculate RMSE
    val_targets = np.concatenate(val_targets, axis=0)
    val_predictions = np.concatenate(val_predictions, axis=0)
    rmse = np.sqrt(mean_squared_error(val_targets, val_predictions))
    return rmse

# Baseline Neural Network Model

In [ ]:
class PricePredictionNN(nn.Module):
    def __init__(self, input_dim, units_layer1=128, units_layer2=64, units_layer3=32):
        super(PricePredictionNN, self).__init__()
        self.model = nn.Sequential(
            nn.Linear(input_dim, units_layer1),
            nn.ReLU(),
            nn.Linear(units_layer1, units_layer2),
            nn.ReLU(),
            nn.Linear(units_layer2, units_layer3),
            nn.ReLU(),
            nn.Linear(units_layer3, 1)  # Single output neuron for regression
        )

    def forward(self, x):
        return self.model(x)

In [ ]:
def objective(trial):
    # Hyperparameter suggestions
    learning_rate = trial.suggest_loguniform('learning_rate', 1e-5, 1e-2)
    weight_decay = trial.suggest_loguniform('weight_decay', 1e-6, 1e-3)
    batch_size = trial.suggest_categorical('batch_size', [32, 64, 128])
    units_layer1 = trial.suggest_int('units_layer1', 64, 256, step=32)
    units_layer2 = trial.suggest_int('units_layer2', 32, 128, step=32)
    units_layer3 = trial.suggest_int('units_layer3', 16, 64, step=16)

    # Initialize the model with the suggested number of units per layer
    model = PricePredictionNN(input_dim, units_layer1, units_layer2, units_layer3).to(device)

    # Define optimizer and loss function
    optimizer = optim.Adam(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
    criterion = nn.MSELoss()

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

    rmse = train_model(model, train_loader, val_loader, criterion, optimizer, epochs=100)

    torch.cuda.empty_cache()

    return rmse

In [ ]:
# Initialize the model, loss function, and optimizer
input_dim = X_train.shape[1]
patience = 10  # Number of epochs to wait for improvement

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Start Optuna study for hyperparameter optimization
study = optuna.create_study(direction='minimize')
study.optimize(objective, n_trials=20)

[I 2024-11-13 01:01:58,860] A new study created in memory with name: no-name-c07651da-f93b-4547-aeb7-d65c90a9e893
<ipython-input-12-8b3723220e46>:3: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  learning_rate = trial.suggest_loguniform('learning_rate', 1e-5, 1e-2)
<ipython-input-12-8b3723220e46>:4: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  weight_decay = trial.suggest_loguniform('weight_decay', 1e-6, 1e-3)


Epoch 1/100, Train Loss: 0.7131, Val Loss: 0.2522
Epoch 2/100, Train Loss: 0.1679, Val Loss: 0.0988
Epoch 3/100, Train Loss: 0.0889, Val Loss: 0.0826
Epoch 4/100, Train Loss: 0.0757, Val Loss: 0.0736
Epoch 5/100, Train Loss: 0.0692, Val Loss: 0.0668
Epoch 6/100, Train Loss: 0.0647, Val Loss: 0.0640
Epoch 7/100, Train Loss: 0.0613, Val Loss: 0.0609
Epoch 8/100, Train Loss: 0.0585, Val Loss: 0.0582
Epoch 9/100, Train Loss: 0.0560, Val Loss: 0.0558
Epoch 10/100, Train Loss: 0.0535, Val Loss: 0.0546
Epoch 11/100, Train Loss: 0.0517, Val Loss: 0.0528
Epoch 12/100, Train Loss: 0.0495, Val Loss: 0.0532
Epoch 13/100, Train Loss: 0.0484, Val Loss: 0.0507
Epoch 14/100, Train Loss: 0.0463, Val Loss: 0.0557
Epoch 15/100, Train Loss: 0.0447, Val Loss: 0.0464
Epoch 16/100, Train Loss: 0.0437, Val Loss: 0.0453
Epoch 17/100, Train Loss: 0.0420, Val Loss: 0.0453
Epoch 18/100, Train Loss: 0.0404, Val Loss: 0.0436
Epoch 19/100, Train Loss: 0.0392, Val Loss: 0.0414
Epoch 20/100, Train Loss: 0.0382, Val Lo

[I 2024-11-13 01:03:33,201] Trial 0 finished with value: 0.14385470747947693 and parameters: {'learning_rate': 6.221329909928104e-05, 'weight_decay': 4.599146132447224e-05, 'batch_size': 64, 'units_layer1': 64, 'units_layer2': 128, 'units_layer3': 48}. Best is trial 0 with value: 0.14385470747947693.


Epoch 100/100, Train Loss: 0.0074, Val Loss: 0.0207


<ipython-input-12-8b3723220e46>:3: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  learning_rate = trial.suggest_loguniform('learning_rate', 1e-5, 1e-2)
<ipython-input-12-8b3723220e46>:4: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  weight_decay = trial.suggest_loguniform('weight_decay', 1e-6, 1e-3)


Epoch 1/100, Train Loss: 0.3415, Val Loss: 0.0744
Epoch 2/100, Train Loss: 0.0719, Val Loss: 0.0602
Epoch 3/100, Train Loss: 0.0584, Val Loss: 0.0522
Epoch 4/100, Train Loss: 0.0525, Val Loss: 0.0511
Epoch 5/100, Train Loss: 0.0473, Val Loss: 0.0446
Epoch 6/100, Train Loss: 0.0415, Val Loss: 0.0406
Epoch 7/100, Train Loss: 0.0405, Val Loss: 0.0371
Epoch 8/100, Train Loss: 0.0336, Val Loss: 0.0353
Epoch 9/100, Train Loss: 0.0356, Val Loss: 0.0315
Epoch 10/100, Train Loss: 0.0272, Val Loss: 0.0294
Epoch 11/100, Train Loss: 0.0245, Val Loss: 0.0290
Epoch 12/100, Train Loss: 0.0234, Val Loss: 0.0303
Epoch 13/100, Train Loss: 0.0224, Val Loss: 0.0284
Epoch 14/100, Train Loss: 0.0209, Val Loss: 0.0277
Epoch 15/100, Train Loss: 0.0214, Val Loss: 0.0268
Epoch 16/100, Train Loss: 0.0168, Val Loss: 0.0291
Epoch 17/100, Train Loss: 0.0173, Val Loss: 0.0260
Epoch 18/100, Train Loss: 0.0162, Val Loss: 0.0239
Epoch 19/100, Train Loss: 0.0144, Val Loss: 0.0258
Epoch 20/100, Train Loss: 0.0128, Val Lo

... (70 outputs left out)

<ipython-input-12-8b3723220e46>:3: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  learning_rate = trial.suggest_loguniform('learning_rate', 1e-5, 1e-2)
<ipython-input-12-8b3723220e46>:4: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  weight_decay = trial.suggest_loguniform('weight_decay', 1e-6, 1e-3)


Epoch 1/100, Train Loss: 0.1142, Val Loss: 0.0728
Epoch 2/100, Train Loss: 0.0651, Val Loss: 0.0432
Epoch 3/100, Train Loss: 0.0537, Val Loss: 0.0384
Epoch 4/100, Train Loss: 0.0428, Val Loss: 0.0441
Epoch 5/100, Train Loss: 0.0454, Val Loss: 0.0303
Epoch 6/100, Train Loss: 0.0358, Val Loss: 0.0510
Epoch 7/100, Train Loss: 0.0324, Val Loss: 0.0232
Epoch 8/100, Train Loss: 0.0212, Val Loss: 0.0279
Epoch 9/100, Train Loss: 0.0227, Val Loss: 0.0227
Epoch 10/100, Train Loss: 0.0237, Val Loss: 0.0285
Epoch 11/100, Train Loss: 0.0181, Val Loss: 0.0524
Epoch 12/100, Train Loss: 0.0277, Val Loss: 0.0373
Epoch 13/100, Train Loss: 0.0220, Val Loss: 0.0212
Epoch 14/100, Train Loss: 0.0200, Val Loss: 0.0225
Epoch 15/100, Train Loss: 0.0152, Val Loss: 0.0256
Epoch 16/100, Train Loss: 0.0178, Val Loss: 0.0275
Epoch 17/100, Train Loss: 0.0237, Val Loss: 0.0232
Epoch 18/100, Train Loss: 0.0165, Val Loss: 0.0225
Epoch 19/100, Train Loss: 0.0168, Val Loss: 0.0217
Epoch 20/100, Train Loss: 0.0164, Val Lo

[I 2024-11-13 01:24:25,801] Trial 19 finished with value: 0.154489204287529 and parameters: {'learning_rate': 0.0017242509950783707, 'weight_decay': 9.613073446361168e-05, 'batch_size': 32, 'units_layer1': 96, 'units_layer2': 128, 'units_layer3': 16}. Best is trial 16 with value: 0.13407495617866516.


Epoch 23/100, Train Loss: 0.0164, Val Loss: 0.0239
Early stopping triggered after 23 epochs.


In [ ]:
# Print the best parameters and best score
print("Best hyperparameters:", study.best_params)
print("Best RMSE:", study.best_value)

Best hyperparameters: {'learning_rate': 0.00015035020966498256, 'weight_decay': 8.428082932204744e-05, 'batch_size': 32, 'units_layer1': 192, 'units_layer2': 128, 'units_layer3': 16}
Best RMSE: 0.13407495617866516


In [ ]:
# Initialize the model with the suggested number of units per layer
units_layer1 = study.best_params['units_layer1']
units_layer2 = study.best_params['units_layer2']
units_layer3 = study.best_params['units_layer3']
learning_rate = study.best_params['learning_rate']
weight_decay = study.best_params['weight_decay']
batch_size = study.best_params['batch_size']

model = PricePredictionNN(input_dim, units_layer1, units_layer2, units_layer3).to(device)

# Define optimizer and loss function
optimizer = optim.Adam(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
criterion = nn.MSELoss()

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

rmse = train_model(model, train_loader, val_loader, criterion, optimizer, epochs=100)

Epoch 1/100, Train Loss: 0.2062, Val Loss: 0.0695
Epoch 2/100, Train Loss: 0.0626, Val Loss: 0.0599
Epoch 3/100, Train Loss: 0.0541, Val Loss: 0.0549
Epoch 4/100, Train Loss: 0.0481, Val Loss: 0.0509
Epoch 5/100, Train Loss: 0.0441, Val Loss: 0.0434
Epoch 6/100, Train Loss: 0.0396, Val Loss: 0.0427
Epoch 7/100, Train Loss: 0.0356, Val Loss: 0.0379
Epoch 8/100, Train Loss: 0.0309, Val Loss: 0.0381
Epoch 9/100, Train Loss: 0.0290, Val Loss: 0.0366
Epoch 10/100, Train Loss: 0.0267, Val Loss: 0.0302
Epoch 11/100, Train Loss: 0.0252, Val Loss: 0.0284
Epoch 12/100, Train Loss: 0.0225, Val Loss: 0.0321
Epoch 13/100, Train Loss: 0.0205, Val Loss: 0.0262
Epoch 14/100, Train Loss: 0.0176, Val Loss: 0.0248
Epoch 15/100, Train Loss: 0.0178, Val Loss: 0.0249
Epoch 16/100, Train Loss: 0.0170, Val Loss: 0.0237
Epoch 17/100, Train Loss: 0.0156, Val Loss: 0.0283
Epoch 18/100, Train Loss: 0.0149, Val Loss: 0.0225
Epoch 19/100, Train Loss: 0.0141, Val Loss: 0.0228
Epoch 20/100, Train Loss: 0.0128, Val Lo

In [ ]:
model.eval()

# Make predictions on the validation set
with torch.no_grad():
    val_predictions = model(X_val_tensor.to(device)).cpu().numpy().flatten()
    val_targets = y_val_tensor.numpy().flatten()

# Calculate RMSE on the validation set (converting back from normalized to original scale)
val_predictions_original_scale = scaler_y.inverse_transform(val_predictions.reshape(-1, 1)).flatten()
val_targets_original_scale = scaler_y.inverse_transform(val_targets.reshape(-1, 1)).flatten()

# Calculate RMSE
rmse = np.sqrt(mean_squared_error(val_targets_original_scale, val_predictions_original_scale))
print(f"Validation RMSE: {rmse:.4f}")

Validation RMSE: 21365.8496


In [ ]:
# Set 1st model as best model first
best_model = model
best_rmse = rmse
best_hyperparameters = study.best_params

# Training with a Larger model

In [ ]:
# Larger model
class PricePredictionLarger(nn.Module):
    def __init__(self, input_dim, units_layer1=256, units_layer2=128, units_layer3=64, units_layer4=32):
        super(PricePredictionLarger, self).__init__()
        self.model = nn.Sequential(
            nn.Linear(input_dim, units_layer1),
            nn.ReLU(),
            nn.Linear(units_layer1, units_layer2),
            nn.ReLU(),
            nn.Linear(units_layer2, units_layer3),
            nn.ReLU(),
            nn.Linear(units_layer3, units_layer4),
            nn.ReLU(),
            nn.Linear(units_layer4, 1)  # Single output neuron for regression
        )

    def forward(self, x):
        return self.model(x)

In [ ]:
def objective_larger(trial):
    # Hyperparameter suggestions
    learning_rate = trial.suggest_loguniform('learning_rate', 1e-5, 1e-2)
    weight_decay = trial.suggest_loguniform('weight_decay', 1e-6, 1e-3)
    batch_size = trial.suggest_categorical('batch_size', [32, 64, 128])
    units_layer1 = trial.suggest_int('units_layer1', 128, 512, step=64)
    units_layer2 = trial.suggest_int('units_layer2', 64, 256, step=32)
    units_layer3 = trial.suggest_int('units_layer3', 32, 128, step=32)
    units_layer4 = trial.suggest_int('units_layer4', 16, 64, step=16)

    # Initialize the model with the suggested number of units per layer
    model = PricePredictionLarger(input_dim, units_layer1, units_layer2, units_layer3, units_layer4).to(device)

    # Define optimizer and loss function
    optimizer = optim.Adam(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
    criterion = nn.MSELoss()

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

    rmse = train_model(model, train_loader, val_loader, criterion, optimizer, epochs=100)

    torch.cuda.empty_cache()

    return rmse

In [ ]:
study = optuna.create_study(direction='minimize')
study.optimize(objective_larger, n_trials=20)

[I 2024-11-13 01:25:48,905] A new study created in memory with name: no-name-f25f49bd-70e5-4992-bdcb-97b753a2956a
<ipython-input-19-ca7e5441454a>:3: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  learning_rate = trial.suggest_loguniform('learning_rate', 1e-5, 1e-2)
<ipython-input-19-ca7e5441454a>:4: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  weight_decay = trial.suggest_loguniform('weight_decay', 1e-6, 1e-3)


Epoch 1/100, Train Loss: 0.2844, Val Loss: 0.0689
Epoch 2/100, Train Loss: 0.0675, Val Loss: 0.0581
Epoch 3/100, Train Loss: 0.0556, Val Loss: 0.0479
Epoch 4/100, Train Loss: 0.0493, Val Loss: 0.0631
Epoch 5/100, Train Loss: 0.0433, Val Loss: 0.0382
Epoch 6/100, Train Loss: 0.0360, Val Loss: 0.0356
Epoch 7/100, Train Loss: 0.0340, Val Loss: 0.0410
Epoch 8/100, Train Loss: 0.0279, Val Loss: 0.0301
Epoch 9/100, Train Loss: 0.0246, Val Loss: 0.0288
Epoch 10/100, Train Loss: 0.0230, Val Loss: 0.0244
Epoch 11/100, Train Loss: 0.0193, Val Loss: 0.0251
Epoch 12/100, Train Loss: 0.0185, Val Loss: 0.0231
Epoch 13/100, Train Loss: 0.0162, Val Loss: 0.0328
Epoch 14/100, Train Loss: 0.0163, Val Loss: 0.0240
Epoch 15/100, Train Loss: 0.0156, Val Loss: 0.0267
Epoch 16/100, Train Loss: 0.0138, Val Loss: 0.0211
Epoch 17/100, Train Loss: 0.0123, Val Loss: 0.0202
Epoch 18/100, Train Loss: 0.0128, Val Loss: 0.0251
Epoch 19/100, Train Loss: 0.0127, Val Loss: 0.0234
Epoch 20/100, Train Loss: 0.0120, Val Lo

[I 2024-11-13 01:26:32,177] Trial 0 finished with value: 0.132851243019104 and parameters: {'learning_rate': 0.00014723167688587786, 'weight_decay': 9.438528042771342e-05, 'batch_size': 64, 'units_layer1': 320, 'units_layer2': 96, 'units_layer3': 128, 'units_layer4': 64}. Best is trial 0 with value: 0.132851243019104.


Epoch 52/100, Train Loss: 0.0061, Val Loss: 0.0176
Early stopping triggered after 52 epochs.


<ipython-input-19-ca7e5441454a>:3: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  learning_rate = trial.suggest_loguniform('learning_rate', 1e-5, 1e-2)
<ipython-input-19-ca7e5441454a>:4: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  weight_decay = trial.suggest_loguniform('weight_decay', 1e-6, 1e-3)


Epoch 1/100, Train Loss: 0.1203, Val Loss: 0.0621
Epoch 2/100, Train Loss: 0.0555, Val Loss: 0.0502
Epoch 3/100, Train Loss: 0.0563, Val Loss: 0.0555
Epoch 4/100, Train Loss: 0.0368, Val Loss: 0.0336
Epoch 5/100, Train Loss: 0.0286, Val Loss: 0.0258
Epoch 6/100, Train Loss: 0.0250, Val Loss: 0.0291
Epoch 7/100, Train Loss: 0.0259, Val Loss: 0.0219
Epoch 8/100, Train Loss: 0.0210, Val Loss: 0.0287
Epoch 9/100, Train Loss: 0.0145, Val Loss: 0.0441
Epoch 10/100, Train Loss: 0.0208, Val Loss: 0.0199
Epoch 11/100, Train Loss: 0.0197, Val Loss: 0.0218
Epoch 12/100, Train Loss: 0.0181, Val Loss: 0.0217
Epoch 13/100, Train Loss: 0.0129, Val Loss: 0.0203
Epoch 14/100, Train Loss: 0.0176, Val Loss: 0.0182
Epoch 15/100, Train Loss: 0.0134, Val Loss: 0.0204
Epoch 16/100, Train Loss: 0.0128, Val Loss: 0.0533
Epoch 17/100, Train Loss: 0.0205, Val Loss: 0.0187
Epoch 18/100, Train Loss: 0.0164, Val Loss: 0.0205
Epoch 19/100, Train Loss: 0.0103, Val Loss: 0.0257
Epoch 20/100, Train Loss: 0.0113, Val Lo

... (70 outputs left out)

<ipython-input-19-ca7e5441454a>:3: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  learning_rate = trial.suggest_loguniform('learning_rate', 1e-5, 1e-2)
<ipython-input-19-ca7e5441454a>:4: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  weight_decay = trial.suggest_loguniform('weight_decay', 1e-6, 1e-3)


Epoch 1/100, Train Loss: 0.5244, Val Loss: 0.0929
Epoch 2/100, Train Loss: 0.0832, Val Loss: 0.0715
Epoch 3/100, Train Loss: 0.0686, Val Loss: 0.0627
Epoch 4/100, Train Loss: 0.0620, Val Loss: 0.0578
Epoch 5/100, Train Loss: 0.0570, Val Loss: 0.0528
Epoch 6/100, Train Loss: 0.0526, Val Loss: 0.0498
Epoch 7/100, Train Loss: 0.0491, Val Loss: 0.0482
Epoch 8/100, Train Loss: 0.0449, Val Loss: 0.0487
Epoch 9/100, Train Loss: 0.0431, Val Loss: 0.0420
Epoch 10/100, Train Loss: 0.0384, Val Loss: 0.0407
Epoch 11/100, Train Loss: 0.0361, Val Loss: 0.0378
Epoch 12/100, Train Loss: 0.0346, Val Loss: 0.0398
Epoch 13/100, Train Loss: 0.0309, Val Loss: 0.0349
Epoch 14/100, Train Loss: 0.0309, Val Loss: 0.0446
Epoch 15/100, Train Loss: 0.0275, Val Loss: 0.0315
Epoch 16/100, Train Loss: 0.0253, Val Loss: 0.0324
Epoch 17/100, Train Loss: 0.0251, Val Loss: 0.0293
Epoch 18/100, Train Loss: 0.0231, Val Loss: 0.0285
Epoch 19/100, Train Loss: 0.0210, Val Loss: 0.0284
Epoch 20/100, Train Loss: 0.0215, Val Lo

[I 2024-11-13 01:49:11,227] Trial 19 finished with value: 0.13912267982959747 and parameters: {'learning_rate': 7.602219201796994e-05, 'weight_decay': 1.091138808331928e-05, 'batch_size': 128, 'units_layer1': 320, 'units_layer2': 224, 'units_layer3': 96, 'units_layer4': 48}. Best is trial 3 with value: 0.13047277927398682.


Epoch 100/100, Train Loss: 0.0037, Val Loss: 0.0194


In [ ]:
print("Best hyperparameters:", study.best_params)
print("Best RMSE:", study.best_value)

Best hyperparameters: {'learning_rate': 7.449621038869098e-05, 'weight_decay': 2.2845857715498996e-06, 'batch_size': 32, 'units_layer1': 512, 'units_layer2': 192, 'units_layer3': 32, 'units_layer4': 16}
Best RMSE: 0.13047277927398682


In [ ]:
# Initialize the model with the suggested number of units per layer
units_layer1 = study.best_params['units_layer1']
units_layer2 = study.best_params['units_layer2']
units_layer3 = study.best_params['units_layer3']
units_layer4 = study.best_params['units_layer4']
learning_rate = study.best_params['learning_rate']
weight_decay = study.best_params['weight_decay']
batch_size = study.best_params['batch_size']

model = PricePredictionLarger(input_dim, units_layer1, units_layer2, units_layer3, units_layer4).to(device)

# Define optimizer and loss function
optimizer = optim.Adam(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
criterion = nn.MSELoss()

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

rmse = train_model(model, train_loader, val_loader, criterion, optimizer, epochs=100)

Epoch 1/100, Train Loss: 0.3078, Val Loss: 0.0751
Epoch 2/100, Train Loss: 0.0649, Val Loss: 0.0549
Epoch 3/100, Train Loss: 0.0573, Val Loss: 0.0540
Epoch 4/100, Train Loss: 0.0506, Val Loss: 0.0461
Epoch 5/100, Train Loss: 0.0468, Val Loss: 0.0418
Epoch 6/100, Train Loss: 0.0408, Val Loss: 0.0384
Epoch 7/100, Train Loss: 0.0356, Val Loss: 0.0340
Epoch 8/100, Train Loss: 0.0333, Val Loss: 0.0327
Epoch 9/100, Train Loss: 0.0287, Val Loss: 0.0303
Epoch 10/100, Train Loss: 0.0272, Val Loss: 0.0319
Epoch 11/100, Train Loss: 0.0242, Val Loss: 0.0264
Epoch 12/100, Train Loss: 0.0212, Val Loss: 0.0298
Epoch 13/100, Train Loss: 0.0191, Val Loss: 0.0256
Epoch 14/100, Train Loss: 0.0195, Val Loss: 0.0271
Epoch 15/100, Train Loss: 0.0179, Val Loss: 0.0226
Epoch 16/100, Train Loss: 0.0165, Val Loss: 0.0283
Epoch 17/100, Train Loss: 0.0158, Val Loss: 0.0254
Epoch 18/100, Train Loss: 0.0152, Val Loss: 0.0219
Epoch 19/100, Train Loss: 0.0128, Val Loss: 0.0230
Epoch 20/100, Train Loss: 0.0139, Val Lo

In [ ]:
model.eval()

# Make predictions on the validation set
with torch.no_grad():
    val_predictions = model(X_val_tensor.to(device)).cpu().numpy().flatten()
    val_targets = y_val_tensor.numpy().flatten()

# Calculate RMSE on the validation set (converting back from normalized to original scale)
val_predictions_original_scale = scaler_y.inverse_transform(val_predictions.reshape(-1, 1)).flatten()
val_targets_original_scale = scaler_y.inverse_transform(val_targets.reshape(-1, 1)).flatten()

# Calculate RMSE
rmse = np.sqrt(mean_squared_error(val_targets_original_scale, val_predictions_original_scale))
print(f"Validation RMSE: {rmse:.4f}")

Validation RMSE: 20828.1270


In [ ]:
if rmse < best_rmse:
    best_model = model
    best_rmse = rmse
    best_hyperparameters = study.best_params

# Training with a Smaller model

In [ ]:
# Smaller model
class PricePredictionSmaller(nn.Module):
    def __init__(self, input_dim, units_layer1=64, units_layer2=32):
        super(PricePredictionSmaller, self).__init__()
        self.model = nn.Sequential(
            nn.Linear(input_dim, units_layer1),
            nn.ReLU(),
            nn.Linear(units_layer1, units_layer2),
            nn.ReLU(),
            nn.Linear(units_layer2, 1)  # Single output neuron for regression
        )

    def forward(self, x):
        return self.model(x)

In [ ]:
def objective_smaller(trial):
    # Hyperparameter suggestions
    learning_rate = trial.suggest_loguniform('learning_rate', 1e-5, 1e-2)
    weight_decay = trial.suggest_loguniform('weight_decay', 1e-6, 1e-3)
    batch_size = trial.suggest_categorical('batch_size', [32, 64, 128])
    units_layer1 = trial.suggest_int('units_layer1', 32, 128, step=16)
    units_layer2 = trial.suggest_int('units_layer2', 16, 64, step=16)

    # Initialize the model with the suggested number of units per layer
    model = PricePredictionSmaller(input_dim, units_layer1, units_layer2).to(device)

    # Define optimizer and loss function
    optimizer = optim.Adam(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
    criterion = nn.MSELoss()

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

    rmse = train_model(model, train_loader, val_loader, criterion, optimizer, epochs=100)

    torch.cuda.empty_cache()

    return rmse

In [ ]:
study_smaller = optuna.create_study(direction='minimize')
study_smaller.optimize(objective_smaller, n_trials=20)

[I 2024-11-13 01:50:47,000] A new study created in memory with name: no-name-b49ed8ae-7c9e-4e6e-864d-f6a54c85be95
<ipython-input-26-77e41df39fb6>:3: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  learning_rate = trial.suggest_loguniform('learning_rate', 1e-5, 1e-2)
<ipython-input-26-77e41df39fb6>:4: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  weight_decay = trial.suggest_loguniform('weight_decay', 1e-6, 1e-3)


Epoch 1/100, Train Loss: 0.8121, Val Loss: 0.5024
Epoch 2/100, Train Loss: 0.4305, Val Loss: 0.2081
Epoch 3/100, Train Loss: 0.1799, Val Loss: 0.1139
Epoch 4/100, Train Loss: 0.1088, Val Loss: 0.0894
Epoch 5/100, Train Loss: 0.0869, Val Loss: 0.0797
Epoch 6/100, Train Loss: 0.0778, Val Loss: 0.0749
Epoch 7/100, Train Loss: 0.0727, Val Loss: 0.0713
Epoch 8/100, Train Loss: 0.0693, Val Loss: 0.0683
Epoch 9/100, Train Loss: 0.0663, Val Loss: 0.0659
Epoch 10/100, Train Loss: 0.0640, Val Loss: 0.0640
Epoch 11/100, Train Loss: 0.0622, Val Loss: 0.0621
Epoch 12/100, Train Loss: 0.0604, Val Loss: 0.0605
Epoch 13/100, Train Loss: 0.0589, Val Loss: 0.0597
Epoch 14/100, Train Loss: 0.0578, Val Loss: 0.0592
Epoch 15/100, Train Loss: 0.0563, Val Loss: 0.0569
Epoch 16/100, Train Loss: 0.0552, Val Loss: 0.0559
Epoch 17/100, Train Loss: 0.0543, Val Loss: 0.0556
Epoch 18/100, Train Loss: 0.0532, Val Loss: 0.0539
Epoch 19/100, Train Loss: 0.0522, Val Loss: 0.0533
Epoch 20/100, Train Loss: 0.0517, Val Lo

[I 2024-11-13 01:51:31,403] Trial 0 finished with value: 0.16129598021507263 and parameters: {'learning_rate': 9.180587387127259e-05, 'weight_decay': 6.321758280960914e-06, 'batch_size': 128, 'units_layer1': 48, 'units_layer2': 32}. Best is trial 0 with value: 0.16129598021507263.


Epoch 100/100, Train Loss: 0.0185, Val Loss: 0.0260


<ipython-input-26-77e41df39fb6>:3: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  learning_rate = trial.suggest_loguniform('learning_rate', 1e-5, 1e-2)
<ipython-input-26-77e41df39fb6>:4: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  weight_decay = trial.suggest_loguniform('weight_decay', 1e-6, 1e-3)


Epoch 1/100, Train Loss: 0.1192, Val Loss: 0.0753
Epoch 2/100, Train Loss: 0.0782, Val Loss: 0.0627
Epoch 3/100, Train Loss: 0.0708, Val Loss: 0.0494
Epoch 4/100, Train Loss: 0.0585, Val Loss: 0.0352
Epoch 5/100, Train Loss: 0.0475, Val Loss: 0.0483
Epoch 6/100, Train Loss: 0.0498, Val Loss: 0.0326
Epoch 7/100, Train Loss: 0.0420, Val Loss: 0.0444
Epoch 8/100, Train Loss: 0.0486, Val Loss: 0.0392
Epoch 9/100, Train Loss: 0.0418, Val Loss: 0.0360
Epoch 10/100, Train Loss: 0.0371, Val Loss: 0.0473
Epoch 11/100, Train Loss: 0.0374, Val Loss: 0.0305
Epoch 12/100, Train Loss: 0.0352, Val Loss: 0.0454
Epoch 13/100, Train Loss: 0.0321, Val Loss: 0.0291
Epoch 14/100, Train Loss: 0.0342, Val Loss: 0.0327
Epoch 15/100, Train Loss: 0.0276, Val Loss: 0.1089
Epoch 16/100, Train Loss: 0.0317, Val Loss: 0.0302
Epoch 17/100, Train Loss: 0.0304, Val Loss: 0.0301
Epoch 18/100, Train Loss: 0.0229, Val Loss: 0.0343
Epoch 19/100, Train Loss: 0.0322, Val Loss: 0.0425
Epoch 20/100, Train Loss: 0.0294, Val Lo

... (70 outputs left out)

<ipython-input-26-77e41df39fb6>:3: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  learning_rate = trial.suggest_loguniform('learning_rate', 1e-5, 1e-2)
<ipython-input-26-77e41df39fb6>:4: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  weight_decay = trial.suggest_loguniform('weight_decay', 1e-6, 1e-3)


Epoch 1/100, Train Loss: 0.1353, Val Loss: 0.0646
Epoch 2/100, Train Loss: 0.0546, Val Loss: 0.0477
Epoch 3/100, Train Loss: 0.0472, Val Loss: 0.0746
Epoch 4/100, Train Loss: 0.0362, Val Loss: 0.0296
Epoch 5/100, Train Loss: 0.0304, Val Loss: 0.0359
Epoch 6/100, Train Loss: 0.0278, Val Loss: 0.0307
Epoch 7/100, Train Loss: 0.0226, Val Loss: 0.0275
Epoch 8/100, Train Loss: 0.0228, Val Loss: 0.0251
Epoch 9/100, Train Loss: 0.0222, Val Loss: 0.0225
Epoch 10/100, Train Loss: 0.0169, Val Loss: 0.0256
Epoch 11/100, Train Loss: 0.0175, Val Loss: 0.0593
Epoch 12/100, Train Loss: 0.0175, Val Loss: 0.0221
Epoch 13/100, Train Loss: 0.0162, Val Loss: 0.0210
Epoch 14/100, Train Loss: 0.0179, Val Loss: 0.0335
Epoch 15/100, Train Loss: 0.0153, Val Loss: 0.0206
Epoch 16/100, Train Loss: 0.0125, Val Loss: 0.0234
Epoch 17/100, Train Loss: 0.0144, Val Loss: 0.0200
Epoch 18/100, Train Loss: 0.0117, Val Loss: 0.0235
Epoch 19/100, Train Loss: 0.0172, Val Loss: 0.0240
Epoch 20/100, Train Loss: 0.0118, Val Lo

[I 2024-11-13 02:05:16,415] Trial 19 finished with value: 0.14035886526107788 and parameters: {'learning_rate': 0.0013204418680320782, 'weight_decay': 1.010584837484506e-05, 'batch_size': 32, 'units_layer1': 64, 'units_layer2': 64}. Best is trial 17 with value: 0.1313258558511734.


Epoch 33/100, Train Loss: 0.0109, Val Loss: 0.0197
Early stopping triggered after 33 epochs.


In [ ]:
print("Best hyperparameters for PricePredictionSmaller:", study_smaller.best_params)
print("Best RMSE for PricePredictionSmaller:", study_smaller.best_value)

Best hyperparameters for PricePredictionSmaller: {'learning_rate': 0.0013948230718259522, 'weight_decay': 1.700106011970867e-05, 'batch_size': 128, 'units_layer1': 96, 'units_layer2': 64}
Best RMSE for PricePredictionSmaller: 0.1313258558511734


In [ ]:
# Initialize the model with the suggested number of units per layer
units_layer1 = study_smaller.best_params['units_layer1']
units_layer2 = study_smaller.best_params['units_layer2']
learning_rate = study_smaller.best_params['learning_rate']
weight_decay = study_smaller.best_params['weight_decay']
batch_size = study_smaller.best_params['batch_size']

model = PricePredictionSmaller(input_dim, units_layer1, units_layer2).to(device)

# Define optimizer and loss function
optimizer = optim.Adam(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
criterion = nn.MSELoss()

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

rmse = train_model(model, train_loader, val_loader, criterion, optimizer, epochs=100)

Epoch 1/100, Train Loss: 0.1354, Val Loss: 0.0574
Epoch 2/100, Train Loss: 0.0603, Val Loss: 0.0489
Epoch 3/100, Train Loss: 0.0516, Val Loss: 0.0447
Epoch 4/100, Train Loss: 0.0461, Val Loss: 0.0468
Epoch 5/100, Train Loss: 0.0394, Val Loss: 0.0369
Epoch 6/100, Train Loss: 0.0325, Val Loss: 0.0314
Epoch 7/100, Train Loss: 0.0304, Val Loss: 0.0368
Epoch 8/100, Train Loss: 0.0231, Val Loss: 0.0307
Epoch 9/100, Train Loss: 0.0212, Val Loss: 0.0298
Epoch 10/100, Train Loss: 0.0219, Val Loss: 0.0253
Epoch 11/100, Train Loss: 0.0160, Val Loss: 0.0269
Epoch 12/100, Train Loss: 0.0162, Val Loss: 0.0253
Epoch 13/100, Train Loss: 0.0153, Val Loss: 0.0279
Epoch 14/100, Train Loss: 0.0147, Val Loss: 0.0253
Epoch 15/100, Train Loss: 0.0142, Val Loss: 0.0245
Epoch 16/100, Train Loss: 0.0129, Val Loss: 0.0215
Epoch 17/100, Train Loss: 0.0114, Val Loss: 0.0214
Epoch 18/100, Train Loss: 0.0106, Val Loss: 0.0244
Epoch 19/100, Train Loss: 0.0170, Val Loss: 0.0228
Epoch 20/100, Train Loss: 0.0125, Val Lo

In [ ]:
model.eval()

# Make predictions on the validation set
with torch.no_grad():
    val_predictions = model(X_val_tensor.to(device)).cpu().numpy().flatten()
    val_targets = y_val_tensor.numpy().flatten()

# Calculate RMSE on the validation set (converting back from normalized to original scale)
val_predictions_original_scale = scaler_y.inverse_transform(val_predictions.reshape(-1, 1)).flatten()
val_targets_original_scale = scaler_y.inverse_transform(val_targets.reshape(-1, 1)).flatten()

# Calculate RMSE
rmse = np.sqrt(mean_squared_error(val_targets_original_scale, val_predictions_original_scale))
print(f"Validation RMSE: {rmse:.4f}")

Validation RMSE: 25111.3340


In [ ]:
if rmse < best_rmse:
    best_model = model
    best_rmse = rmse
    best_hyperparameters = study_smaller.best_params

# Using a different activation function (LeakyReLU)

In [ ]:
# Different activation functions
class PricePredictionLeakyRelu(nn.Module):
    def __init__(self, input_dim, units_layer1=128, units_layer2=64, units_layer3=32, leaky_slope=0.1):
        super(PricePredictionLeakyRelu, self).__init__()
        self.model = nn.Sequential(
            nn.Linear(input_dim, units_layer1),
            nn.LeakyReLU(negative_slope=leaky_slope),
            nn.Linear(units_layer1, units_layer2),
            nn.LeakyReLU(negative_slope=leaky_slope),
            nn.Linear(units_layer2, units_layer3),
            nn.LeakyReLU(negative_slope=leaky_slope),
            nn.Linear(units_layer3, 1)  # Single output neuron for regression
        )

    def forward(self, x):
        return self.model(x)

In [ ]:
def objective_leakyrelu(trial):
    # Hyperparameter suggestions
    learning_rate = trial.suggest_loguniform('learning_rate', 1e-5, 1e-2)
    weight_decay = trial.suggest_loguniform('weight_decay', 1e-6, 1e-3)
    batch_size = trial.suggest_categorical('batch_size', [32, 64, 128])
    units_layer1 = trial.suggest_int('units_layer1', 64, 256, step=32)
    units_layer2 = trial.suggest_int('units_layer2', 32, 128, step=32)
    units_layer3 = trial.suggest_int('units_layer3', 16, 64, step=16)
    leaky_slope = trial.suggest_float('leaky_slope', 0.01, 0.3)

    # Initialize the model with the suggested number of units per layer
    model = PricePredictionLeakyRelu(input_dim, units_layer1, units_layer2, units_layer3).to(device)

    # Define optimizer and loss function
    optimizer = optim.Adam(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
    criterion = nn.MSELoss()

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

    rmse = train_model(model, train_loader, val_loader, criterion, optimizer, epochs=100)

    torch.cuda.empty_cache()

    return rmse

In [ ]:
study_leakyrelu = optuna.create_study(direction='minimize')
study_leakyrelu.optimize(objective_leakyrelu, n_trials=20)

[I 2024-11-13 02:05:34,773] A new study created in memory with name: no-name-f9f978e1-7788-45b1-ad9d-48a007b2450e
<ipython-input-33-0918913849e6>:3: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  learning_rate = trial.suggest_loguniform('learning_rate', 1e-5, 1e-2)
<ipython-input-33-0918913849e6>:4: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  weight_decay = trial.suggest_loguniform('weight_decay', 1e-6, 1e-3)


Epoch 1/100, Train Loss: 0.2545, Val Loss: 0.0731
Epoch 2/100, Train Loss: 0.0680, Val Loss: 0.0595
Epoch 3/100, Train Loss: 0.0590, Val Loss: 0.0531
Epoch 4/100, Train Loss: 0.0544, Val Loss: 0.0548
Epoch 5/100, Train Loss: 0.0516, Val Loss: 0.0471
Epoch 6/100, Train Loss: 0.0459, Val Loss: 0.0427
Epoch 7/100, Train Loss: 0.0430, Val Loss: 0.0404
Epoch 8/100, Train Loss: 0.0396, Val Loss: 0.0366
Epoch 9/100, Train Loss: 0.0352, Val Loss: 0.0366
Epoch 10/100, Train Loss: 0.0311, Val Loss: 0.0310
Epoch 11/100, Train Loss: 0.0317, Val Loss: 0.0303
Epoch 12/100, Train Loss: 0.0276, Val Loss: 0.0283
Epoch 13/100, Train Loss: 0.0247, Val Loss: 0.0278
Epoch 14/100, Train Loss: 0.0232, Val Loss: 0.0286
Epoch 15/100, Train Loss: 0.0223, Val Loss: 0.0264
Epoch 16/100, Train Loss: 0.0210, Val Loss: 0.0253
Epoch 17/100, Train Loss: 0.0213, Val Loss: 0.0243
Epoch 18/100, Train Loss: 0.0191, Val Loss: 0.0238
Epoch 19/100, Train Loss: 0.0174, Val Loss: 0.0242
Epoch 20/100, Train Loss: 0.0163, Val Lo

[I 2024-11-13 02:06:19,169] Trial 0 finished with value: 0.13783575594425201 and parameters: {'learning_rate': 0.00016329119980219365, 'weight_decay': 0.0007074066316534427, 'batch_size': 64, 'units_layer1': 224, 'units_layer2': 64, 'units_layer3': 64, 'leaky_slope': 0.2018774832697682}. Best is trial 0 with value: 0.13783575594425201.


Epoch 58/100, Train Loss: 0.0076, Val Loss: 0.0190
Early stopping triggered after 58 epochs.


<ipython-input-33-0918913849e6>:3: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  learning_rate = trial.suggest_loguniform('learning_rate', 1e-5, 1e-2)
<ipython-input-33-0918913849e6>:4: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  weight_decay = trial.suggest_loguniform('weight_decay', 1e-6, 1e-3)


Epoch 1/100, Train Loss: 0.1284, Val Loss: 0.0601
Epoch 2/100, Train Loss: 0.0582, Val Loss: 0.0495
Epoch 3/100, Train Loss: 0.0512, Val Loss: 0.0417
Epoch 4/100, Train Loss: 0.0452, Val Loss: 0.0349
Epoch 5/100, Train Loss: 0.0351, Val Loss: 0.0345
Epoch 6/100, Train Loss: 0.0349, Val Loss: 0.0276
Epoch 7/100, Train Loss: 0.0258, Val Loss: 0.0299
Epoch 8/100, Train Loss: 0.0218, Val Loss: 0.0312
Epoch 9/100, Train Loss: 0.0265, Val Loss: 0.0268
Epoch 10/100, Train Loss: 0.0198, Val Loss: 0.0221
Epoch 11/100, Train Loss: 0.0178, Val Loss: 0.0227
Epoch 12/100, Train Loss: 0.0161, Val Loss: 0.0221
Epoch 13/100, Train Loss: 0.0169, Val Loss: 0.0224
Epoch 14/100, Train Loss: 0.0160, Val Loss: 0.0204
Epoch 15/100, Train Loss: 0.0150, Val Loss: 0.0238
Epoch 16/100, Train Loss: 0.0150, Val Loss: 0.0249
Epoch 17/100, Train Loss: 0.0136, Val Loss: 0.0294
Epoch 18/100, Train Loss: 0.0128, Val Loss: 0.0199
Epoch 19/100, Train Loss: 0.0156, Val Loss: 0.0217
Epoch 20/100, Train Loss: 0.0103, Val Lo

... (70 outputs left out)

<ipython-input-33-0918913849e6>:3: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  learning_rate = trial.suggest_loguniform('learning_rate', 1e-5, 1e-2)
<ipython-input-33-0918913849e6>:4: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  weight_decay = trial.suggest_loguniform('weight_decay', 1e-6, 1e-3)


Epoch 1/100, Train Loss: 1.0099, Val Loss: 0.7695
Epoch 2/100, Train Loss: 0.8576, Val Loss: 0.6153
Epoch 3/100, Train Loss: 0.6364, Val Loss: 0.4079
Epoch 4/100, Train Loss: 0.3926, Val Loss: 0.2318
Epoch 5/100, Train Loss: 0.2275, Val Loss: 0.1507
Epoch 6/100, Train Loss: 0.1538, Val Loss: 0.1198
Epoch 7/100, Train Loss: 0.1204, Val Loss: 0.1038
Epoch 8/100, Train Loss: 0.1030, Val Loss: 0.0936
Epoch 9/100, Train Loss: 0.0920, Val Loss: 0.0862
Epoch 10/100, Train Loss: 0.0847, Val Loss: 0.0809
Epoch 11/100, Train Loss: 0.0796, Val Loss: 0.0769
Epoch 12/100, Train Loss: 0.0760, Val Loss: 0.0744
Epoch 13/100, Train Loss: 0.0732, Val Loss: 0.0716
Epoch 14/100, Train Loss: 0.0709, Val Loss: 0.0697
Epoch 15/100, Train Loss: 0.0691, Val Loss: 0.0679
Epoch 16/100, Train Loss: 0.0673, Val Loss: 0.0667
Epoch 17/100, Train Loss: 0.0658, Val Loss: 0.0648
Epoch 18/100, Train Loss: 0.0644, Val Loss: 0.0640
Epoch 19/100, Train Loss: 0.0632, Val Loss: 0.0627
Epoch 20/100, Train Loss: 0.0619, Val Lo

[I 2024-11-13 02:24:07,705] Trial 19 finished with value: 0.17454443871974945 and parameters: {'learning_rate': 1.8902913601818055e-05, 'weight_decay': 0.0002960850525599893, 'batch_size': 128, 'units_layer1': 224, 'units_layer2': 64, 'units_layer3': 32, 'leaky_slope': 0.2704667847757149}. Best is trial 12 with value: 0.1356123983860016.


Epoch 100/100, Train Loss: 0.0265, Val Loss: 0.0305


In [ ]:
print("Best hyperparameters for PricePredictionLeakyRelu:", study_leakyrelu.best_params)
print("Best RMSE for PricePredictionLeakyRelu:", study_leakyrelu.best_value)

Best hyperparameters for PricePredictionLeakyRelu: {'learning_rate': 0.0003051206861008539, 'weight_decay': 0.00029867720664764926, 'batch_size': 64, 'units_layer1': 192, 'units_layer2': 64, 'units_layer3': 64, 'leaky_slope': 0.17001611889356408}
Best RMSE for PricePredictionLeakyRelu: 0.1356123983860016


In [ ]:
# Initialize the model with the suggested number of units per layer
units_layer1 = study_leakyrelu.best_params['units_layer1']
units_layer2 = study_leakyrelu.best_params['units_layer2']
units_layer3 = study_leakyrelu.best_params['units_layer3']
learning_rate = study_leakyrelu.best_params['learning_rate']
weight_decay = study_leakyrelu.best_params['weight_decay']
batch_size = study_leakyrelu.best_params['batch_size']
leaky_slope = study_leakyrelu.best_params['leaky_slope']

model = PricePredictionLeakyRelu(input_dim, units_layer1, units_layer2, units_layer3).to(device)

# Define optimizer and loss function
optimizer = optim.Adam(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
criterion = nn.MSELoss()

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

rmse = train_model(model, train_loader, val_loader, criterion, optimizer, epochs=100)

Epoch 1/100, Train Loss: 0.2109, Val Loss: 0.0626
Epoch 2/100, Train Loss: 0.0594, Val Loss: 0.0507
Epoch 3/100, Train Loss: 0.0500, Val Loss: 0.0491
Epoch 4/100, Train Loss: 0.0468, Val Loss: 0.0420
Epoch 5/100, Train Loss: 0.0380, Val Loss: 0.0385
Epoch 6/100, Train Loss: 0.0342, Val Loss: 0.0317
Epoch 7/100, Train Loss: 0.0291, Val Loss: 0.0317
Epoch 8/100, Train Loss: 0.0273, Val Loss: 0.0293
Epoch 9/100, Train Loss: 0.0255, Val Loss: 0.0265
Epoch 10/100, Train Loss: 0.0212, Val Loss: 0.0257
Epoch 11/100, Train Loss: 0.0205, Val Loss: 0.0227
Epoch 12/100, Train Loss: 0.0183, Val Loss: 0.0313
Epoch 13/100, Train Loss: 0.0177, Val Loss: 0.0229
Epoch 14/100, Train Loss: 0.0164, Val Loss: 0.0231
Epoch 15/100, Train Loss: 0.0155, Val Loss: 0.0227
Epoch 16/100, Train Loss: 0.0155, Val Loss: 0.0228
Epoch 17/100, Train Loss: 0.0144, Val Loss: 0.0221
Epoch 18/100, Train Loss: 0.0142, Val Loss: 0.0205
Epoch 19/100, Train Loss: 0.0119, Val Loss: 0.0224
Epoch 20/100, Train Loss: 0.0131, Val Lo

In [ ]:
model.eval()

# Make predictions on the validation set
with torch.no_grad():
    val_predictions = model(X_val_tensor.to(device)).cpu().numpy().flatten()
    val_targets = y_val_tensor.numpy().flatten()

# Calculate RMSE on the validation set (converting back from normalized to original scale)
val_predictions_original_scale = scaler_y.inverse_transform(val_predictions.reshape(-1, 1)).flatten()
val_targets_original_scale = scaler_y.inverse_transform(val_targets.reshape(-1, 1)).flatten()

# Calculate RMSE
rmse = np.sqrt(mean_squared_error(val_targets_original_scale, val_predictions_original_scale))
print(f"Validation RMSE: {rmse:.4f}")

Validation RMSE: 21760.8750


In [ ]:
if rmse < best_rmse:
    best_model = model
    best_rmse = rmse
    best_hyperparameters = study_leakyrelu.best_params

# Adding Dropouts to the model

In [ ]:
class PricePredictionDropouts(nn.Module):
    def __init__(self, input_dim, units_layer1=128, units_layer2=64, units_layer3=32, dropout_rate=0.2):
        super(PricePredictionDropouts, self).__init__()
        self.model = nn.Sequential(
            nn.Linear(input_dim, units_layer1),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(units_layer1, units_layer2),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(units_layer2, units_layer3),
            nn.ReLU(),
            nn.Linear(units_layer3, 1)  # Single output neuron for regression
        )

    def forward(self, x):
        return self.model(x)

In [ ]:
def objective_dropouts(trial):
    # Hyperparameter suggestions
    learning_rate = trial.suggest_loguniform('learning_rate', 1e-5, 1e-2)
    weight_decay = trial.suggest_loguniform('weight_decay', 1e-6, 1e-3)
    batch_size = trial.suggest_categorical('batch_size', [32, 64, 128])
    units_layer1 = trial.suggest_int('units_layer1', 64, 256, step=32)
    units_layer2 = trial.suggest_int('units_layer2', 32, 128, step=32)
    units_layer3 = trial.suggest_int('units_layer3', 16, 64, step=16)
    dropout_rate = trial.suggest_float('dropout_rate', 0.1, 0.5)

    # Initialize the model with suggested hyperparameters
    model = PricePredictionDropouts(input_dim, units_layer1, units_layer2, units_layer3, dropout_rate).to(device)

    # Define optimizer and loss function
    optimizer = optim.Adam(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
    criterion = nn.MSELoss()

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

    rmse = train_model(model, train_loader, val_loader, criterion, optimizer, epochs=100)

    torch.cuda.empty_cache()

    return rmse

In [ ]:
study_dropouts = optuna.create_study(direction='minimize')
study_dropouts.optimize(objective_dropouts, n_trials=20)

[I 2024-11-13 02:25:00,175] A new study created in memory with name: no-name-deafca3f-5d70-4273-86ec-75f50c9d23e4
<ipython-input-40-738fc1586cba>:3: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  learning_rate = trial.suggest_loguniform('learning_rate', 1e-5, 1e-2)
<ipython-input-40-738fc1586cba>:4: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  weight_decay = trial.suggest_loguniform('weight_decay', 1e-6, 1e-3)


Epoch 1/100, Train Loss: 0.2101, Val Loss: 0.0703
Epoch 2/100, Train Loss: 0.1630, Val Loss: 0.0629
Epoch 3/100, Train Loss: 0.1238, Val Loss: 0.0667
Epoch 4/100, Train Loss: 0.1244, Val Loss: 0.0863
Epoch 5/100, Train Loss: 0.1302, Val Loss: 0.0579
Epoch 6/100, Train Loss: 0.1025, Val Loss: 0.0523
Epoch 7/100, Train Loss: 0.1033, Val Loss: 0.0776
Epoch 8/100, Train Loss: 0.0980, Val Loss: 0.0520
Epoch 9/100, Train Loss: 0.1450, Val Loss: 0.0550
Epoch 10/100, Train Loss: 0.1110, Val Loss: 0.0435
Epoch 11/100, Train Loss: 0.0998, Val Loss: 0.0686
Epoch 12/100, Train Loss: 0.1010, Val Loss: 0.0649
Epoch 13/100, Train Loss: 0.0827, Val Loss: 0.0782
Epoch 14/100, Train Loss: 0.1011, Val Loss: 0.0403
Epoch 15/100, Train Loss: 0.0952, Val Loss: 0.0631
Epoch 16/100, Train Loss: 0.0925, Val Loss: 0.0462
Epoch 17/100, Train Loss: 0.0840, Val Loss: 0.0693
Epoch 18/100, Train Loss: 0.0994, Val Loss: 0.0592
Epoch 19/100, Train Loss: 0.0870, Val Loss: 0.0565
Epoch 20/100, Train Loss: 0.0952, Val Lo

[I 2024-11-13 02:25:11,744] Trial 0 finished with value: 0.24003203213214874 and parameters: {'learning_rate': 0.005232006509215511, 'weight_decay': 5.405010198117966e-05, 'batch_size': 128, 'units_layer1': 64, 'units_layer2': 128, 'units_layer3': 16, 'dropout_rate': 0.39448031584653787}. Best is trial 0 with value: 0.24003203213214874.


Epoch 24/100, Train Loss: 0.0764, Val Loss: 0.0576
Early stopping triggered after 24 epochs.


<ipython-input-40-738fc1586cba>:3: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  learning_rate = trial.suggest_loguniform('learning_rate', 1e-5, 1e-2)
<ipython-input-40-738fc1586cba>:4: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  weight_decay = trial.suggest_loguniform('weight_decay', 1e-6, 1e-3)


Epoch 1/100, Train Loss: 1.0000, Val Loss: 0.7756
Epoch 2/100, Train Loss: 0.9080, Val Loss: 0.6810
Epoch 3/100, Train Loss: 0.7880, Val Loss: 0.5532
Epoch 4/100, Train Loss: 0.6225, Val Loss: 0.4087
Epoch 5/100, Train Loss: 0.4671, Val Loss: 0.2887
Epoch 6/100, Train Loss: 0.3418, Val Loss: 0.2137
Epoch 7/100, Train Loss: 0.2720, Val Loss: 0.1739
Epoch 8/100, Train Loss: 0.2339, Val Loss: 0.1473
Epoch 9/100, Train Loss: 0.1940, Val Loss: 0.1303
Epoch 10/100, Train Loss: 0.1680, Val Loss: 0.1182
Epoch 11/100, Train Loss: 0.1605, Val Loss: 0.1087
Epoch 12/100, Train Loss: 0.1477, Val Loss: 0.1016
Epoch 13/100, Train Loss: 0.1431, Val Loss: 0.0965
Epoch 14/100, Train Loss: 0.1456, Val Loss: 0.0923
Epoch 15/100, Train Loss: 0.1382, Val Loss: 0.0892
Epoch 16/100, Train Loss: 0.1301, Val Loss: 0.0860
Epoch 17/100, Train Loss: 0.1325, Val Loss: 0.0835
Epoch 18/100, Train Loss: 0.1281, Val Loss: 0.0816
Epoch 19/100, Train Loss: 0.1196, Val Loss: 0.0798
Epoch 20/100, Train Loss: 0.1189, Val Lo

... (70 outputs left out)

<ipython-input-40-738fc1586cba>:3: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  learning_rate = trial.suggest_loguniform('learning_rate', 1e-5, 1e-2)
<ipython-input-40-738fc1586cba>:4: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  weight_decay = trial.suggest_loguniform('weight_decay', 1e-6, 1e-3)


Epoch 1/100, Train Loss: 0.2298, Val Loss: 0.0762
Epoch 2/100, Train Loss: 0.0935, Val Loss: 0.0600
Epoch 3/100, Train Loss: 0.0818, Val Loss: 0.0539
Epoch 4/100, Train Loss: 0.0729, Val Loss: 0.0698
Epoch 5/100, Train Loss: 0.0676, Val Loss: 0.0497
Epoch 6/100, Train Loss: 0.0633, Val Loss: 0.0434
Epoch 7/100, Train Loss: 0.0627, Val Loss: 0.0568
Epoch 8/100, Train Loss: 0.0609, Val Loss: 0.0400
Epoch 9/100, Train Loss: 0.0617, Val Loss: 0.0370
Epoch 10/100, Train Loss: 0.0551, Val Loss: 0.0412
Epoch 11/100, Train Loss: 0.0516, Val Loss: 0.0339
Epoch 12/100, Train Loss: 0.0493, Val Loss: 0.0463
Epoch 13/100, Train Loss: 0.0490, Val Loss: 0.0304
Epoch 14/100, Train Loss: 0.0462, Val Loss: 0.0375
Epoch 15/100, Train Loss: 0.0441, Val Loss: 0.0389
Epoch 16/100, Train Loss: 0.0444, Val Loss: 0.0385
Epoch 17/100, Train Loss: 0.0412, Val Loss: 0.0342
Epoch 18/100, Train Loss: 0.0380, Val Loss: 0.0346
Epoch 19/100, Train Loss: 0.0396, Val Loss: 0.0254
Epoch 20/100, Train Loss: 0.0415, Val Lo

[I 2024-11-13 02:38:21,120] Trial 19 finished with value: 0.20277762413024902 and parameters: {'learning_rate': 0.0002089010349120207, 'weight_decay': 1.894704412175376e-05, 'batch_size': 32, 'units_layer1': 224, 'units_layer2': 64, 'units_layer3': 64, 'dropout_rate': 0.23618619634294086}. Best is trial 10 with value: 0.14321564137935638.


Epoch 39/100, Train Loss: 0.0246, Val Loss: 0.0411
Early stopping triggered after 39 epochs.


In [ ]:
# Print the best parameters and best score
print("Best hyperparameters for PricePredictionDropouts:", study_dropouts.best_params)
print("Best RMSE for PricePredictionDropouts:", study_dropouts.best_value)

Best hyperparameters for PricePredictionDropouts: {'learning_rate': 0.0003569093270239579, 'weight_decay': 1.067324266652831e-05, 'batch_size': 128, 'units_layer1': 224, 'units_layer2': 64, 'units_layer3': 64, 'dropout_rate': 0.10597152842114967}
Best RMSE for PricePredictionDropouts: 0.14321564137935638


In [ ]:
# Initialize the model with the suggested number of units per layer
units_layer1 = study_dropouts.best_params['units_layer1']
units_layer2 = study_dropouts.best_params['units_layer2']
units_layer3 = study_dropouts.best_params['units_layer3']
learning_rate = study_dropouts.best_params['learning_rate']
weight_decay = study_dropouts.best_params['weight_decay']
batch_size = study_dropouts.best_params['batch_size']
dropout_rate = study_dropouts.best_params['dropout_rate']

model = PricePredictionDropouts(input_dim, units_layer1, units_layer2, units_layer3, dropout_rate).to(device)

# Define optimizer and loss function
optimizer = optim.Adam(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
criterion = nn.MSELoss()

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

rmse = train_model(model, train_loader, val_loader, criterion, optimizer, epochs=100)

Epoch 1/100, Train Loss: 0.2874, Val Loss: 0.0738
Epoch 2/100, Train Loss: 0.0826, Val Loss: 0.0595
Epoch 3/100, Train Loss: 0.0743, Val Loss: 0.0742
Epoch 4/100, Train Loss: 0.0631, Val Loss: 0.0555
Epoch 5/100, Train Loss: 0.0565, Val Loss: 0.0437
Epoch 6/100, Train Loss: 0.0571, Val Loss: 0.0397
Epoch 7/100, Train Loss: 0.0527, Val Loss: 0.0396
Epoch 8/100, Train Loss: 0.0477, Val Loss: 0.0334
Epoch 9/100, Train Loss: 0.0442, Val Loss: 0.0376
Epoch 10/100, Train Loss: 0.0438, Val Loss: 0.0303
Epoch 11/100, Train Loss: 0.0401, Val Loss: 0.0353
Epoch 12/100, Train Loss: 0.0369, Val Loss: 0.0375
Epoch 13/100, Train Loss: 0.0373, Val Loss: 0.0306
Epoch 14/100, Train Loss: 0.0323, Val Loss: 0.0279
Epoch 15/100, Train Loss: 0.0346, Val Loss: 0.0252
Epoch 16/100, Train Loss: 0.0331, Val Loss: 0.0248
Epoch 17/100, Train Loss: 0.0286, Val Loss: 0.0246
Epoch 18/100, Train Loss: 0.0276, Val Loss: 0.0248
Epoch 19/100, Train Loss: 0.0270, Val Loss: 0.0229
Epoch 20/100, Train Loss: 0.0273, Val Lo

In [ ]:
model.eval()

# Make predictions on the validation set
with torch.no_grad():
    val_predictions = model(X_val_tensor.to(device)).cpu().numpy().flatten()
    val_targets = y_val_tensor.numpy().flatten()

# Calculate RMSE on the validation set (converting back from normalized to original scale)
val_predictions_original_scale = scaler_y.inverse_transform(val_predictions.reshape(-1, 1)).flatten()
val_targets_original_scale = scaler_y.inverse_transform(val_targets.reshape(-1, 1)).flatten()

# Calculate RMSE
rmse = np.sqrt(mean_squared_error(val_targets_original_scale, val_predictions_original_scale))
print(f"Validation RMSE: {rmse:.4f}")

Validation RMSE: 23952.1758


In [ ]:
if rmse < best_rmse:
    best_model = model
    best_rmse = rmse
    best_hyperparameters = study_dropouts.best_params

# Batch Normalization

In [ ]:
# Batch Normalization
class PricePredictionBatchNorm(nn.Module):
    def __init__(self, input_dim, units_layer1=128, units_layer2=64, units_layer3=32):
        super(PricePredictionBatchNorm, self).__init__()
        self.model = nn.Sequential(
            nn.Linear(input_dim, units_layer1),
            nn.BatchNorm1d(units_layer1),
            nn.ReLU(),
            nn.Linear(units_layer1, units_layer2),
            nn.BatchNorm1d(units_layer2),
            nn.ReLU(),
            nn.Linear(units_layer2, units_layer3),
            nn.BatchNorm1d(units_layer3),
            nn.ReLU(),
            nn.Linear(units_layer3, 1)  # Single output neuron for regression
        )

    def forward(self, x):
        return self.model(x)

In [ ]:
def objective_batchnorm(trial):
    # Hyperparameter suggestions
    learning_rate = trial.suggest_loguniform('learning_rate', 1e-5, 1e-2)
    weight_decay = trial.suggest_loguniform('weight_decay', 1e-6, 1e-3)
    batch_size = trial.suggest_categorical('batch_size', [32, 64, 128])
    units_layer1 = trial.suggest_int('units_layer1', 64, 256, step=32)
    units_layer2 = trial.suggest_int('units_layer2', 32, 128, step=32)
    units_layer3 = trial.suggest_int('units_layer3', 16, 64, step=16)

    # Initialize the model with the suggested hyperparameters
    model = PricePredictionBatchNorm(input_dim, units_layer1, units_layer2, units_layer3).to(device)

    # Define optimizer and loss function
    optimizer = optim.Adam(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
    criterion = nn.MSELoss()

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

    rmse = train_model(model, train_loader, val_loader, criterion, optimizer, epochs=100)

    torch.cuda.empty_cache()

    return rmse

In [ ]:
study_batchnorm = optuna.create_study(direction='minimize')
study_batchnorm.optimize(objective_batchnorm, n_trials=20)

[I 2024-11-13 02:38:42,671] A new study created in memory with name: no-name-12429d09-6d05-4296-9e02-cc58a036d2db
<ipython-input-47-ff1b4ca2471f>:3: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  learning_rate = trial.suggest_loguniform('learning_rate', 1e-5, 1e-2)
<ipython-input-47-ff1b4ca2471f>:4: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  weight_decay = trial.suggest_loguniform('weight_decay', 1e-6, 1e-3)


Epoch 1/100, Train Loss: 0.8239, Val Loss: 0.3827
Epoch 2/100, Train Loss: 0.4526, Val Loss: 0.1825
Epoch 3/100, Train Loss: 0.3517, Val Loss: 0.1190
Epoch 4/100, Train Loss: 0.3216, Val Loss: 0.1177
Epoch 5/100, Train Loss: 0.2987, Val Loss: 0.1130
Epoch 6/100, Train Loss: 0.2773, Val Loss: 0.1099
Epoch 7/100, Train Loss: 0.2671, Val Loss: 0.1005
Epoch 8/100, Train Loss: 0.2591, Val Loss: 0.1088
Epoch 9/100, Train Loss: 0.2389, Val Loss: 0.1089
Epoch 10/100, Train Loss: 0.2239, Val Loss: 0.0970
Epoch 11/100, Train Loss: 0.2164, Val Loss: 0.0905
Epoch 12/100, Train Loss: 0.2212, Val Loss: 0.0830
Epoch 13/100, Train Loss: 0.2032, Val Loss: 0.0789
Epoch 14/100, Train Loss: 0.1880, Val Loss: 0.0751
Epoch 15/100, Train Loss: 0.1846, Val Loss: 0.0859
Epoch 16/100, Train Loss: 0.1712, Val Loss: 0.0779
Epoch 17/100, Train Loss: 0.1623, Val Loss: 0.0736
Epoch 18/100, Train Loss: 0.1526, Val Loss: 0.0628
Epoch 19/100, Train Loss: 0.1574, Val Loss: 0.1009
Epoch 20/100, Train Loss: 0.1443, Val Lo

[I 2024-11-13 02:39:53,134] Trial 0 finished with value: 0.2735915780067444 and parameters: {'learning_rate': 3.868888200700626e-05, 'weight_decay': 0.0002801635413509325, 'batch_size': 32, 'units_layer1': 64, 'units_layer2': 32, 'units_layer3': 32}. Best is trial 0 with value: 0.2735915780067444.


Epoch 40/100, Train Loss: 0.0880, Val Loss: 0.0749
Early stopping triggered after 40 epochs.


<ipython-input-47-ff1b4ca2471f>:3: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  learning_rate = trial.suggest_loguniform('learning_rate', 1e-5, 1e-2)
<ipython-input-47-ff1b4ca2471f>:4: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  weight_decay = trial.suggest_loguniform('weight_decay', 1e-6, 1e-3)


Epoch 1/100, Train Loss: 0.3817, Val Loss: 0.1033
Epoch 2/100, Train Loss: 0.2341, Val Loss: 0.0905
Epoch 3/100, Train Loss: 0.2005, Val Loss: 0.0873
Epoch 4/100, Train Loss: 0.1779, Val Loss: 0.0962
Epoch 5/100, Train Loss: 0.1528, Val Loss: 0.0772
Epoch 6/100, Train Loss: 0.1510, Val Loss: 0.0728
Epoch 7/100, Train Loss: 0.1306, Val Loss: 0.0628
Epoch 8/100, Train Loss: 0.1135, Val Loss: 0.0723
Epoch 9/100, Train Loss: 0.1111, Val Loss: 0.0642
Epoch 10/100, Train Loss: 0.1030, Val Loss: 0.0654
Epoch 11/100, Train Loss: 0.1078, Val Loss: 0.0600
Epoch 12/100, Train Loss: 0.0972, Val Loss: 0.0527
Epoch 13/100, Train Loss: 0.0924, Val Loss: 0.0513
Epoch 14/100, Train Loss: 0.0817, Val Loss: 0.0503
Epoch 15/100, Train Loss: 0.0848, Val Loss: 0.0493
Epoch 16/100, Train Loss: 0.0807, Val Loss: 0.0470
Epoch 17/100, Train Loss: 0.0758, Val Loss: 0.0484
Epoch 18/100, Train Loss: 0.0706, Val Loss: 0.0433
Epoch 19/100, Train Loss: 0.0665, Val Loss: 0.0470
Epoch 20/100, Train Loss: 0.0605, Val Lo

... (70 outputs left out)

<ipython-input-47-ff1b4ca2471f>:3: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  learning_rate = trial.suggest_loguniform('learning_rate', 1e-5, 1e-2)
<ipython-input-47-ff1b4ca2471f>:4: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  weight_decay = trial.suggest_loguniform('weight_decay', 1e-6, 1e-3)


Epoch 1/100, Train Loss: 0.1804, Val Loss: 0.0693
Epoch 2/100, Train Loss: 0.1080, Val Loss: 0.0499
Epoch 3/100, Train Loss: 0.1003, Val Loss: 0.0442
Epoch 4/100, Train Loss: 0.0799, Val Loss: 0.0831
Epoch 5/100, Train Loss: 0.0795, Val Loss: 0.0733
Epoch 6/100, Train Loss: 0.0682, Val Loss: 0.0732
Epoch 7/100, Train Loss: 0.0650, Val Loss: 0.0648
Epoch 8/100, Train Loss: 0.0666, Val Loss: 0.0428
Epoch 9/100, Train Loss: 0.0658, Val Loss: 0.0394
Epoch 10/100, Train Loss: 0.0564, Val Loss: 0.0470
Epoch 11/100, Train Loss: 0.0528, Val Loss: 0.0434
Epoch 12/100, Train Loss: 0.0532, Val Loss: 0.0411
Epoch 13/100, Train Loss: 0.0530, Val Loss: 0.0399
Epoch 14/100, Train Loss: 0.0548, Val Loss: 0.0702
Epoch 15/100, Train Loss: 0.0410, Val Loss: 0.0392
Epoch 16/100, Train Loss: 0.0461, Val Loss: 0.0385
Epoch 17/100, Train Loss: 0.0475, Val Loss: 0.0294
Epoch 18/100, Train Loss: 0.0407, Val Loss: 0.0532
Epoch 19/100, Train Loss: 0.0395, Val Loss: 0.0559
Epoch 20/100, Train Loss: 0.0444, Val Lo

[I 2024-11-13 02:52:49,169] Trial 19 finished with value: 0.17723448574543 and parameters: {'learning_rate': 0.00439932337152438, 'weight_decay': 2.8149155578245374e-06, 'batch_size': 32, 'units_layer1': 160, 'units_layer2': 64, 'units_layer3': 64}. Best is trial 16 with value: 0.147586852312088.


Epoch 39/100, Train Loss: 0.0321, Val Loss: 0.0314
Early stopping triggered after 39 epochs.


In [ ]:
print("Best hyperparameters for PricePredictionBatchNorm:", study_batchnorm.best_params)
print("Best RMSE for PricePredictionBatchNorm:", study_batchnorm.best_value)

Best hyperparameters for PricePredictionBatchNorm: {'learning_rate': 0.0006781358443866796, 'weight_decay': 3.829644192579873e-06, 'batch_size': 128, 'units_layer1': 192, 'units_layer2': 32, 'units_layer3': 48}
Best RMSE for PricePredictionBatchNorm: 0.147586852312088


In [ ]:
# Initialize the model with the suggested number of units per layer
units_layer1 = study_batchnorm.best_params['units_layer1']
units_layer2 = study_batchnorm.best_params['units_layer2']
units_layer3 = study_batchnorm.best_params['units_layer3']
learning_rate = study_batchnorm.best_params['learning_rate']
weight_decay = study_batchnorm.best_params['weight_decay']
batch_size = study_batchnorm.best_params['batch_size']

model = PricePredictionBatchNorm(input_dim, units_layer1, units_layer2, units_layer3).to(device)

# Define optimizer and loss function
optimizer = optim.Adam(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
criterion = nn.MSELoss()

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

rmse = train_model(model, train_loader, val_loader, criterion, optimizer, epochs=100)

Epoch 1/100, Train Loss: 0.1971, Val Loss: 0.0710
Epoch 2/100, Train Loss: 0.1109, Val Loss: 0.0644
Epoch 3/100, Train Loss: 0.0737, Val Loss: 0.0424
Epoch 4/100, Train Loss: 0.0611, Val Loss: 0.0376
Epoch 5/100, Train Loss: 0.0532, Val Loss: 0.0411
Epoch 6/100, Train Loss: 0.0570, Val Loss: 0.0389
Epoch 7/100, Train Loss: 0.0433, Val Loss: 0.0336
Epoch 8/100, Train Loss: 0.0358, Val Loss: 0.0322
Epoch 9/100, Train Loss: 0.0370, Val Loss: 0.0320
Epoch 10/100, Train Loss: 0.0336, Val Loss: 0.0513
Epoch 11/100, Train Loss: 0.0332, Val Loss: 0.0381
Epoch 12/100, Train Loss: 0.0352, Val Loss: 0.0265
Epoch 13/100, Train Loss: 0.0312, Val Loss: 0.0333
Epoch 14/100, Train Loss: 0.0321, Val Loss: 0.0329
Epoch 15/100, Train Loss: 0.0313, Val Loss: 0.0320
Epoch 16/100, Train Loss: 0.0286, Val Loss: 0.0323
Epoch 17/100, Train Loss: 0.0255, Val Loss: 0.0265
Epoch 18/100, Train Loss: 0.0245, Val Loss: 0.0304
Epoch 19/100, Train Loss: 0.0318, Val Loss: 0.0308
Epoch 20/100, Train Loss: 0.0234, Val Lo

In [ ]:
model.eval()

# Make predictions on the validation set
with torch.no_grad():
    val_predictions = model(X_val_tensor.to(device)).cpu().numpy().flatten()
    val_targets = y_val_tensor.numpy().flatten()

# Calculate RMSE on the validation set (converting back from normalized to original scale)
val_predictions_original_scale = scaler_y.inverse_transform(val_predictions.reshape(-1, 1)).flatten()
val_targets_original_scale = scaler_y.inverse_transform(val_targets.reshape(-1, 1)).flatten()

# Calculate RMSE
rmse = np.sqrt(mean_squared_error(val_targets_original_scale, val_predictions_original_scale))
print(f"Validation RMSE: {rmse:.4f}")

Validation RMSE: 24904.7285


In [ ]:
if rmse < best_rmse:
    best_model = model
    best_rmse = rmse
    best_hyperparameters = study_batchnorm.best_params

# Residual Blocks

In [ ]:
# Residual Connections
class ResidualBlock(nn.Module):
    def __init__(self, input_dim, output_dim):
        super(ResidualBlock, self).__init__()
        self.fc = nn.Linear(input_dim, output_dim)
        self.relu = nn.ReLU()

    def forward(self, x):
        out = self.fc(x)
        out = self.relu(out)
        return x + out  # Add residual connection

class PricePredictionResidual(nn.Module):
    def __init__(self, input_dim, units_layer=128, num_residual_blocks=2):
        super(PricePredictionResidual, self).__init__()
        self.input_fc = nn.Linear(input_dim, units_layer)
        self.residual_blocks = nn.ModuleList([ResidualBlock(units_layer, units_layer) for _ in range(num_residual_blocks)])
        self.output_fc = nn.Linear(units_layer, 1)

    def forward(self, x):
        out = self.input_fc(x)
        for residual_block in self.residual_blocks:
            out = residual_block(out)
        out = self.output_fc(out)
        return out

In [ ]:
def objective_residual(trial):
    # Hyperparameter suggestions
    learning_rate = trial.suggest_loguniform('learning_rate', 1e-5, 1e-2)
    weight_decay = trial.suggest_loguniform('weight_decay', 1e-6, 1e-3)
    batch_size = trial.suggest_categorical('batch_size', [32, 64, 128])
    units_layer = trial.suggest_int('units_layer', 64, 256, step=32)
    num_residual_blocks = trial.suggest_int('num_residual_blocks', 1, 4)

    # Initialize the model with the suggested hyperparameters
    model = PricePredictionResidual(input_dim, units_layer, num_residual_blocks).to(device)

    # Define optimizer and loss function
    optimizer = optim.Adam(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
    criterion = nn.MSELoss()

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

    rmse = train_model(model, train_loader, val_loader, criterion, optimizer, epochs=100)

    torch.cuda.empty_cache()

    return rmse

In [ ]:
study_residual = optuna.create_study(direction='minimize')
study_residual.optimize(objective_residual, n_trials=20)

[I 2024-11-13 02:53:23,206] A new study created in memory with name: no-name-23d24063-d471-4682-ab41-f3b59cfd9186
<ipython-input-54-8fa140882300>:3: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  learning_rate = trial.suggest_loguniform('learning_rate', 1e-5, 1e-2)
<ipython-input-54-8fa140882300>:4: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  weight_decay = trial.suggest_loguniform('weight_decay', 1e-6, 1e-3)


Epoch 1/100, Train Loss: 0.0979, Val Loss: 0.0596
Epoch 2/100, Train Loss: 0.0637, Val Loss: 0.0543
Epoch 3/100, Train Loss: 0.0544, Val Loss: 0.0501
Epoch 4/100, Train Loss: 0.0532, Val Loss: 0.0400
Epoch 5/100, Train Loss: 0.0449, Val Loss: 0.0500
Epoch 6/100, Train Loss: 0.0444, Val Loss: 0.0356
Epoch 7/100, Train Loss: 0.0445, Val Loss: 0.0313
Epoch 8/100, Train Loss: 0.0356, Val Loss: 0.0293
Epoch 9/100, Train Loss: 0.0322, Val Loss: 0.0286
Epoch 10/100, Train Loss: 0.0306, Val Loss: 0.0433
Epoch 11/100, Train Loss: 0.0349, Val Loss: 0.0319
Epoch 12/100, Train Loss: 0.0305, Val Loss: 0.0267
Epoch 13/100, Train Loss: 0.0286, Val Loss: 0.0274
Epoch 14/100, Train Loss: 0.0291, Val Loss: 0.0506
Epoch 15/100, Train Loss: 0.0329, Val Loss: 0.0277
Epoch 16/100, Train Loss: 0.0260, Val Loss: 0.0270
Epoch 17/100, Train Loss: 0.0243, Val Loss: 0.0275
Epoch 18/100, Train Loss: 0.0239, Val Loss: 0.0334
Epoch 19/100, Train Loss: 0.0231, Val Loss: 0.0289
Epoch 20/100, Train Loss: 0.0250, Val Lo

[I 2024-11-13 02:54:46,535] Trial 0 finished with value: 0.14677563309669495 and parameters: {'learning_rate': 0.0003822152208483638, 'weight_decay': 2.4799733739947382e-05, 'batch_size': 32, 'units_layer': 96, 'num_residual_blocks': 1}. Best is trial 0 with value: 0.14677563309669495.


Epoch 67/100, Train Loss: 0.0110, Val Loss: 0.0215
Early stopping triggered after 67 epochs.


<ipython-input-54-8fa140882300>:3: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  learning_rate = trial.suggest_loguniform('learning_rate', 1e-5, 1e-2)
<ipython-input-54-8fa140882300>:4: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  weight_decay = trial.suggest_loguniform('weight_decay', 1e-6, 1e-3)


Epoch 1/100, Train Loss: 0.1453, Val Loss: 0.0692
Epoch 2/100, Train Loss: 0.0700, Val Loss: 0.0580
Epoch 3/100, Train Loss: 0.0581, Val Loss: 0.0551
Epoch 4/100, Train Loss: 0.0525, Val Loss: 0.0468
Epoch 5/100, Train Loss: 0.0501, Val Loss: 0.0448
Epoch 6/100, Train Loss: 0.0497, Val Loss: 0.0410
Epoch 7/100, Train Loss: 0.0416, Val Loss: 0.0454
Epoch 8/100, Train Loss: 0.0408, Val Loss: 0.0396
Epoch 9/100, Train Loss: 0.0385, Val Loss: 0.0333
Epoch 10/100, Train Loss: 0.0334, Val Loss: 0.0364
Epoch 11/100, Train Loss: 0.0324, Val Loss: 0.0375
Epoch 12/100, Train Loss: 0.0311, Val Loss: 0.0403
Epoch 13/100, Train Loss: 0.0315, Val Loss: 0.0429
Epoch 14/100, Train Loss: 0.0294, Val Loss: 0.0431
Epoch 15/100, Train Loss: 0.0317, Val Loss: 0.0727
Epoch 16/100, Train Loss: 0.0333, Val Loss: 0.0288
Epoch 17/100, Train Loss: 0.0260, Val Loss: 0.0252
Epoch 18/100, Train Loss: 0.0235, Val Loss: 0.0247
Epoch 19/100, Train Loss: 0.0292, Val Loss: 0.0240
Epoch 20/100, Train Loss: 0.0206, Val Lo

... (70 outputs left out)

<ipython-input-54-8fa140882300>:3: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  learning_rate = trial.suggest_loguniform('learning_rate', 1e-5, 1e-2)
<ipython-input-54-8fa140882300>:4: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  weight_decay = trial.suggest_loguniform('weight_decay', 1e-6, 1e-3)


Epoch 1/100, Train Loss: 0.3333, Val Loss: 0.1008
Epoch 2/100, Train Loss: 0.0872, Val Loss: 0.0791
Epoch 3/100, Train Loss: 0.0720, Val Loss: 0.0693
Epoch 4/100, Train Loss: 0.0646, Val Loss: 0.0605
Epoch 5/100, Train Loss: 0.0576, Val Loss: 0.0570
Epoch 6/100, Train Loss: 0.0542, Val Loss: 0.0527
Epoch 7/100, Train Loss: 0.0499, Val Loss: 0.0495
Epoch 8/100, Train Loss: 0.0467, Val Loss: 0.0485
Epoch 9/100, Train Loss: 0.0449, Val Loss: 0.0464
Epoch 10/100, Train Loss: 0.0410, Val Loss: 0.0443
Epoch 11/100, Train Loss: 0.0384, Val Loss: 0.0406
Epoch 12/100, Train Loss: 0.0374, Val Loss: 0.0390
Epoch 13/100, Train Loss: 0.0344, Val Loss: 0.0406
Epoch 14/100, Train Loss: 0.0335, Val Loss: 0.0369
Epoch 15/100, Train Loss: 0.0319, Val Loss: 0.0348
Epoch 16/100, Train Loss: 0.0290, Val Loss: 0.0353
Epoch 17/100, Train Loss: 0.0281, Val Loss: 0.0339
Epoch 18/100, Train Loss: 0.0262, Val Loss: 0.0353
Epoch 19/100, Train Loss: 0.0263, Val Loss: 0.0334
Epoch 20/100, Train Loss: 0.0250, Val Lo

[I 2024-11-13 03:13:49,290] Trial 19 finished with value: 0.14395621418952942 and parameters: {'learning_rate': 2.7560526621404315e-05, 'weight_decay': 7.396694070030201e-05, 'batch_size': 128, 'units_layer': 256, 'num_residual_blocks': 3}. Best is trial 6 with value: 0.13686907291412354.


Epoch 100/100, Train Loss: 0.0061, Val Loss: 0.0207


In [ ]:
# Print the best parameters and best score
print("Best hyperparameters for PricePredictionResidual:", study_residual.best_params)
print("Best RMSE for PricePredictionResidual:", study_residual.best_value)

Best hyperparameters for PricePredictionResidual: {'learning_rate': 0.00014853506685304847, 'weight_decay': 1.570646067586395e-06, 'batch_size': 32, 'units_layer': 160, 'num_residual_blocks': 3}
Best RMSE for PricePredictionResidual: 0.13686907291412354


In [ ]:
# Initialize the model with the suggested number of units per layer
units_layer = study_residual.best_params['units_layer']
learning_rate = study_residual.best_params['learning_rate']
weight_decay = study_residual.best_params['weight_decay']
batch_size = study_residual.best_params['batch_size']
num_residual_blocks = study_residual.best_params['num_residual_blocks']

model = PricePredictionResidual(input_dim, units_layer, num_residual_blocks).to(device)

# Define optimizer and loss function
optimizer = optim.Adam(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
criterion = nn.MSELoss()

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

rmse = train_model(model, train_loader, val_loader, criterion, optimizer, epochs=100)

Epoch 1/100, Train Loss: 0.1106, Val Loss: 0.0596
Epoch 2/100, Train Loss: 0.0695, Val Loss: 0.0738
Epoch 3/100, Train Loss: 0.0685, Val Loss: 0.0395
Epoch 4/100, Train Loss: 0.0392, Val Loss: 0.0361
Epoch 5/100, Train Loss: 0.0365, Val Loss: 0.0345
Epoch 6/100, Train Loss: 0.0356, Val Loss: 0.0785
Epoch 7/100, Train Loss: 0.0379, Val Loss: 0.0275
Epoch 8/100, Train Loss: 0.0249, Val Loss: 0.0278
Epoch 9/100, Train Loss: 0.0300, Val Loss: 0.0331
Epoch 10/100, Train Loss: 0.0236, Val Loss: 0.0281
Epoch 11/100, Train Loss: 0.0244, Val Loss: 0.0417
Epoch 12/100, Train Loss: 0.0171, Val Loss: 0.0345
Epoch 13/100, Train Loss: 0.0176, Val Loss: 0.0419
Epoch 14/100, Train Loss: 0.0180, Val Loss: 0.0234
Epoch 15/100, Train Loss: 0.0198, Val Loss: 0.0258
Epoch 16/100, Train Loss: 0.0172, Val Loss: 0.0247
Epoch 17/100, Train Loss: 0.0151, Val Loss: 0.0240
Epoch 18/100, Train Loss: 0.0151, Val Loss: 0.0313
Epoch 19/100, Train Loss: 0.0153, Val Loss: 0.0240
Epoch 20/100, Train Loss: 0.0136, Val Lo

In [ ]:
model.eval()

# Make predictions on the validation set
with torch.no_grad():
    val_predictions = model(X_val_tensor.to(device)).cpu().numpy().flatten()
    val_targets = y_val_tensor.numpy().flatten()

# Calculate RMSE on the validation set (converting back from normalized to original scale)
val_predictions_original_scale = scaler_y.inverse_transform(val_predictions.reshape(-1, 1)).flatten()
val_targets_original_scale = scaler_y.inverse_transform(val_targets.reshape(-1, 1)).flatten()

# Calculate RMSE
rmse = np.sqrt(mean_squared_error(val_targets_original_scale, val_predictions_original_scale))
print(f"Validation RMSE: {rmse:.4f}")

Validation RMSE: 28504.1426


In [ ]:
if rmse < best_rmse:
    best_model = model
    best_rmse = rmse
    best_hyperparameters = study_residual.best_params

# Gated Linear Units

In [ ]:
# Gated Linear Units
class PricePredictionGLU(nn.Module):
    def __init__(self, input_dim, units_layer1=128, units_layer2=64):
        super(PricePredictionGLU, self).__init__()
        self.fc1 = nn.Linear(input_dim, units_layer1)
        self.glu1 = nn.GLU()
        self.fc2 = nn.Linear(units_layer1 // 2, units_layer2)  # GLU halves the output size
        self.glu2 = nn.GLU()
        self.output = nn.Linear(units_layer2 // 2, 1)  # Single output for regression

    def forward(self, x):
        x = self.glu1(self.fc1(x))
        x = self.glu2(self.fc2(x))
        return self.output(x)

In [ ]:
def objective_glu(trial):
    # Hyperparameter suggestions
    learning_rate = trial.suggest_loguniform('learning_rate', 1e-5, 1e-2)
    weight_decay = trial.suggest_loguniform('weight_decay', 1e-6, 1e-3)
    batch_size = trial.suggest_categorical('batch_size', [32, 64, 128])
    units_layer1 = trial.suggest_int('units_layer1', 64, 256, step=32)
    units_layer2 = trial.suggest_int('units_layer2', 32, 128, step=32)

    # Initialize the model with the suggested hyperparameters
    model = PricePredictionGLU(input_dim, units_layer1, units_layer2).to(device)

    # Define optimizer and loss function
    optimizer = optim.Adam(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
    criterion = nn.MSELoss()

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

    rmse = train_model(model, train_loader, val_loader, criterion, optimizer, epochs=100)

    torch.cuda.empty_cache()

    return rmse

In [ ]:
study_glu = optuna.create_study(direction='minimize')
study_glu.optimize(objective_glu, n_trials=20)

[I 2024-11-13 03:14:50,830] A new study created in memory with name: no-name-a903b313-a937-4939-946a-d00ad4807065
<ipython-input-61-d7e721c6476c>:3: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  learning_rate = trial.suggest_loguniform('learning_rate', 1e-5, 1e-2)
<ipython-input-61-d7e721c6476c>:4: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  weight_decay = trial.suggest_loguniform('weight_decay', 1e-6, 1e-3)


Epoch 1/100, Train Loss: 0.1106, Val Loss: 0.0637
Epoch 2/100, Train Loss: 0.0646, Val Loss: 0.0475
Epoch 3/100, Train Loss: 0.0463, Val Loss: 0.0394
Epoch 4/100, Train Loss: 0.0393, Val Loss: 0.0323
Epoch 5/100, Train Loss: 0.0300, Val Loss: 0.0312
Epoch 6/100, Train Loss: 0.0292, Val Loss: 0.0302
Epoch 7/100, Train Loss: 0.0237, Val Loss: 0.0248
Epoch 8/100, Train Loss: 0.0195, Val Loss: 0.0341
Epoch 9/100, Train Loss: 0.0221, Val Loss: 0.0258
Epoch 10/100, Train Loss: 0.0179, Val Loss: 0.0239
Epoch 11/100, Train Loss: 0.0181, Val Loss: 0.0260
Epoch 12/100, Train Loss: 0.0173, Val Loss: 0.0209
Epoch 13/100, Train Loss: 0.0162, Val Loss: 0.0212
Epoch 14/100, Train Loss: 0.0158, Val Loss: 0.0339
Epoch 15/100, Train Loss: 0.0161, Val Loss: 0.0198
Epoch 16/100, Train Loss: 0.0133, Val Loss: 0.0208
Epoch 17/100, Train Loss: 0.0150, Val Loss: 0.0229
Epoch 18/100, Train Loss: 0.0125, Val Loss: 0.0211
Epoch 19/100, Train Loss: 0.0142, Val Loss: 0.0198
Epoch 20/100, Train Loss: 0.0121, Val Lo

[I 2024-11-13 03:15:26,991] Trial 0 finished with value: 0.14244599640369415 and parameters: {'learning_rate': 0.0017537748106281196, 'weight_decay': 4.416044047037073e-05, 'batch_size': 64, 'units_layer1': 192, 'units_layer2': 96}. Best is trial 0 with value: 0.14244599640369415.


Epoch 50/100, Train Loss: 0.0085, Val Loss: 0.0203
Early stopping triggered after 50 epochs.


<ipython-input-61-d7e721c6476c>:3: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  learning_rate = trial.suggest_loguniform('learning_rate', 1e-5, 1e-2)
<ipython-input-61-d7e721c6476c>:4: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  weight_decay = trial.suggest_loguniform('weight_decay', 1e-6, 1e-3)


Epoch 1/100, Train Loss: 0.0946, Val Loss: 0.0513
Epoch 2/100, Train Loss: 0.0625, Val Loss: 0.0580
Epoch 3/100, Train Loss: 0.0498, Val Loss: 0.0524
Epoch 4/100, Train Loss: 0.0384, Val Loss: 0.0341
Epoch 5/100, Train Loss: 0.0365, Val Loss: 0.0297
Epoch 6/100, Train Loss: 0.0274, Val Loss: 0.0344
Epoch 7/100, Train Loss: 0.0271, Val Loss: 0.0361
Epoch 8/100, Train Loss: 0.0299, Val Loss: 0.0276
Epoch 9/100, Train Loss: 0.0196, Val Loss: 0.0231
Epoch 10/100, Train Loss: 0.0212, Val Loss: 0.0330
Epoch 11/100, Train Loss: 0.0234, Val Loss: 0.0242
Epoch 12/100, Train Loss: 0.0191, Val Loss: 0.0246
Epoch 13/100, Train Loss: 0.0191, Val Loss: 0.0300
Epoch 14/100, Train Loss: 0.0168, Val Loss: 0.0231
Epoch 15/100, Train Loss: 0.0179, Val Loss: 0.0210
Epoch 16/100, Train Loss: 0.0158, Val Loss: 0.0221
Epoch 17/100, Train Loss: 0.0143, Val Loss: 0.0240
Epoch 18/100, Train Loss: 0.0133, Val Loss: 0.0220
Epoch 19/100, Train Loss: 0.0164, Val Loss: 0.0223
Epoch 20/100, Train Loss: 0.0145, Val Lo

... (70 outputs left out)

<ipython-input-61-d7e721c6476c>:3: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  learning_rate = trial.suggest_loguniform('learning_rate', 1e-5, 1e-2)
<ipython-input-61-d7e721c6476c>:4: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  weight_decay = trial.suggest_loguniform('weight_decay', 1e-6, 1e-3)


Epoch 1/100, Train Loss: 0.4469, Val Loss: 0.0970
Epoch 2/100, Train Loss: 0.0884, Val Loss: 0.0792
Epoch 3/100, Train Loss: 0.0774, Val Loss: 0.0731
Epoch 4/100, Train Loss: 0.0707, Val Loss: 0.0674
Epoch 5/100, Train Loss: 0.0658, Val Loss: 0.0636
Epoch 6/100, Train Loss: 0.0613, Val Loss: 0.0603
Epoch 7/100, Train Loss: 0.0577, Val Loss: 0.0576
Epoch 8/100, Train Loss: 0.0558, Val Loss: 0.0554
Epoch 9/100, Train Loss: 0.0524, Val Loss: 0.0540
Epoch 10/100, Train Loss: 0.0504, Val Loss: 0.0542
Epoch 11/100, Train Loss: 0.0487, Val Loss: 0.0509
Epoch 12/100, Train Loss: 0.0468, Val Loss: 0.0503
Epoch 13/100, Train Loss: 0.0456, Val Loss: 0.0490
Epoch 14/100, Train Loss: 0.0439, Val Loss: 0.0491
Epoch 15/100, Train Loss: 0.0428, Val Loss: 0.0480
Epoch 16/100, Train Loss: 0.0411, Val Loss: 0.0460
Epoch 17/100, Train Loss: 0.0400, Val Loss: 0.0457
Epoch 18/100, Train Loss: 0.0391, Val Loss: 0.0443
Epoch 19/100, Train Loss: 0.0381, Val Loss: 0.0439
Epoch 20/100, Train Loss: 0.0369, Val Lo

[I 2024-11-13 03:34:30,756] Trial 19 finished with value: 0.15237566828727722 and parameters: {'learning_rate': 7.849434292578706e-05, 'weight_decay': 2.961773138779287e-06, 'batch_size': 32, 'units_layer1': 128, 'units_layer2': 128}. Best is trial 14 with value: 0.12669837474822998.


Epoch 100/100, Train Loss: 0.0110, Val Loss: 0.0232


In [ ]:
# Print the best parameters and best score
print("Best hyperparameters for PricePredictionGLU:", study_glu.best_params)
print("Best RMSE for PricePredictionGLU:", study_glu.best_value)

Best hyperparameters for PricePredictionGLU: {'learning_rate': 0.0006844811149551601, 'weight_decay': 1.4609451050465885e-05, 'batch_size': 32, 'units_layer1': 160, 'units_layer2': 128}
Best RMSE for PricePredictionGLU: 0.12669837474822998


In [ ]:
# Initialize the model with the suggested number of units per layer
units_layer1 = study_glu.best_params['units_layer1']
units_layer2 = study_glu.best_params['units_layer2']
learning_rate = study_glu.best_params['learning_rate']
weight_decay = study_glu.best_params['weight_decay']
batch_size = study_glu.best_params['batch_size']

model = PricePredictionGLU(input_dim, units_layer1, units_layer2).to(device)

# Define optimizer and loss function
optimizer = optim.Adam(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
criterion = nn.MSELoss()

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

rmse = train_model(model, train_loader, val_loader, criterion, optimizer, epochs=100)

Epoch 1/100, Train Loss: 0.1071, Val Loss: 0.0580
Epoch 2/100, Train Loss: 0.0656, Val Loss: 0.0470
Epoch 3/100, Train Loss: 0.0493, Val Loss: 0.0506
Epoch 4/100, Train Loss: 0.0390, Val Loss: 0.0362
Epoch 5/100, Train Loss: 0.0355, Val Loss: 0.0364
Epoch 6/100, Train Loss: 0.0324, Val Loss: 0.0423
Epoch 7/100, Train Loss: 0.0310, Val Loss: 0.0326
Epoch 8/100, Train Loss: 0.0236, Val Loss: 0.0277
Epoch 9/100, Train Loss: 0.0206, Val Loss: 0.0334
Epoch 10/100, Train Loss: 0.0225, Val Loss: 0.0447
Epoch 11/100, Train Loss: 0.0203, Val Loss: 0.0261
Epoch 12/100, Train Loss: 0.0178, Val Loss: 0.0245
Epoch 13/100, Train Loss: 0.0161, Val Loss: 0.0244
Epoch 14/100, Train Loss: 0.0146, Val Loss: 0.0236
Epoch 15/100, Train Loss: 0.0182, Val Loss: 0.0228
Epoch 16/100, Train Loss: 0.0136, Val Loss: 0.0240
Epoch 17/100, Train Loss: 0.0136, Val Loss: 0.0250
Epoch 18/100, Train Loss: 0.0133, Val Loss: 0.0222
Epoch 19/100, Train Loss: 0.0132, Val Loss: 0.0227
Epoch 20/100, Train Loss: 0.0105, Val Lo

In [ ]:
model.eval()

# Make predictions on the validation set
with torch.no_grad():
    val_predictions = model(X_val_tensor.to(device)).cpu().numpy().flatten()
    val_targets = y_val_tensor.numpy().flatten()

# Calculate RMSE on the validation set (converting back from normalized to original scale)
val_predictions_original_scale = scaler_y.inverse_transform(val_predictions.reshape(-1, 1)).flatten()
val_targets_original_scale = scaler_y.inverse_transform(val_targets.reshape(-1, 1)).flatten()

# Calculate RMSE
rmse = np.sqrt(mean_squared_error(val_targets_original_scale, val_predictions_original_scale))
print(f"Validation RMSE: {rmse:.4f}")

Validation RMSE: 22385.3691


In [ ]:
if rmse < best_rmse:
    best_model = model
    best_rmse = rmse
    best_hyperparameters = study_glu.best_params

# Generate Predictions using best model based on Validation RMSE

In [ ]:
X_test_normalized = scaler_X.fit_transform(test_df.values)
X_test_tensor = torch.tensor(X_test_normalized, dtype=torch.float32)
test_dataset = TensorDataset(X_test_tensor)
test_loader = DataLoader(test_dataset, batch_size=best_hyperparameters['batch_size'], shuffle=False)

In [ ]:
predictions = []
best_model.eval()
with torch.no_grad():
    for inputs in test_loader:
        outputs = best_model(inputs[0].to(device))
        predictions.extend(outputs.cpu().numpy().flatten())

In [ ]:
predictions = scaler_y.inverse_transform(np.array(predictions).reshape(-1, 1)).flatten()

In [ ]:
test_results = pd.DataFrame({'Id': range(len(predictions)), 'Predicted': predictions})

In [ ]:
test_results.to_csv('submission.csv', index=False)